# 第 65 课｜PPO 消融、故障归因与训练报告审计

**不要同时改 GAE、Critic、KL 和 batching，然后只报一个最终得分。**

上一课建立多训练种子、开发选择与独立留出的可复现协议。本课进一步追问：哪个机制真的改变了实验？是模型训练问题、评价方法问题，还是更新预算发生了变化？

本课交付完全独立的 Jupyter 讲义与 Python 脚本，内置三状态两动作 Actor–Critic、PPO-Clip、GAE、value loss、KL early stop、mini/microbatch、训练日志与重评分逻辑。

**环境：Python 3.10+、PyTorch、CPU双精度、单线程，离线。** 没有预训练语言模型、TRL或CUDA/GPU性能测量。

本课重点是**实验方法**：多训练种子逐因素比较、识别批次预算混杂、构造交互反例、冻结开发选择，核对训练记录和权重。少量教学任务/种子不能支持广泛算法排名。

## 0．本课要建立的四个习惯

| 问题 | 必须留下的证据 |
|---|---|
| 改的究竟是哪一个因素？ | 明确的基线、逐字段 diff 和类别登记 |
| 两个模型是否在可比较的训练预算上？ | 环境新transition、重复呈现次数、Actor/Critic optimizer step 分别计数 |
| 更高分数是在开发还是留出？ | 冻结开发选择文件，留出使用不同**训练** seed |
| 报告中的每个分数能否重算？ | 按试验键保存 trace、checkpoint、seed、模型参数和摘要，并从权重重新推理 |

这门课不会用“一个结果差0.01%”直接推断哪种算法全局更好；实际留出区间只是描述性不确定性。多次评估episode不等于独立训练run。[Deep RL That Matters](https://ojs.aaai.org/index.php/AAAI/article/view/11694)、[rliable](https://github.com/google-research/rliable)

建议先完成 1–6 节，再运行完整对照，最后做练习与故障检测。

### 实现单元：固定环境、配置与内容摘要

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第65课：PPO消融实验、故障归因与训练报告审计。

Python 3.10+、PyTorch；CPU三状态两动作PPO教学Actor/Critic。
成对多种子逐因素改动GAE、value loss、KL早停与batching；
开发种子锁定选择，留出训练种子独立评估。
微批等价性是数值检查，不是GPU性能声明。
这不是预训练LLM、真实RL大规模benchmark或因果普遍性结论。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需已有PyTorch环境；请使用匹配的Jupyter内核") from exc

VERSION = "lesson65-ablation-audit-v1"
SOURCES = {
    "ppo_paper": "https://arxiv.org/abs/1707.06347",
    "spinningup_ppo": "https://spinningup.openai.com/en/latest/algorithms/ppo.html",
    "spinningup_code": "https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html",
    "sb3_code": "https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html",
    "gae": "https://arxiv.org/abs/1506.02438",
    "gymnasium_time_limit": "https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/",
    "pytorch_autograd": "https://docs.pytorch.org/docs/stable/notes/autograd.html",
    "statistical_precipice": "https://arxiv.org/abs/2108.13264",
    "rliable": "https://github.com/google-research/rliable",
    "deep_rl_matters": "https://ojs.aaai.org/index.php/AAAI/article/view/11694",
    "implementation_matters": "https://arxiv.org/abs/2005.12729",
    "sb3_rl_tips": "https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html",
}

class LessonError(ValueError):
    """本课程协议错误；不静默修改任务定义。"""

def require(condition, message):
    if not bool(condition):
        raise LessonError(message)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def sha256(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def tensor_hash(named):
    h = hashlib.sha256()
    for name, tensor in sorted(named.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()

def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)

@dataclass(frozen=True)
class Config:
    seed: int = 6301
    gamma: float = .95
    gae_lambda: float = .95
    horizon: int = 6
    rollouts: int = 24
    episodes_per_rollout: int = 25
    epochs: int = 3
    minibatch_size: int = 17
    microbatch_size: int = 5
    actor_lr: float = .012
    critic_lr: float = .010
    actor_hidden: int = 24
    critic_hidden: int = 24
    clip_ratio: float = .2
    value_clip: float = .2
    entropy_coef: float = .006
    target_kl: float = .06
    grad_clip: float = 1.
    eval_episodes: int = 120
    value_method: str = "clipmax"
    bootstrap_truncation: bool = True
    kl_stop_enabled: bool = True

    def validate(self):
        for name in ('seed','horizon','rollouts','episodes_per_rollout','epochs',
                     'minibatch_size','microbatch_size','actor_hidden','critic_hidden','eval_episodes'):
            v=getattr(self,name)
            require(type(v) is int and v >= (0 if name=='seed' else 1),name+'必须为整数')
        require(3<=self.horizon<=16 and 1<=self.epochs<=8, 'horizon/epochs超出教学范围')
        require(2<=self.minibatch_size<=512 and 1<=self.microbatch_size<=self.minibatch_size,
                'microbatch必须小于等于minibatch')
        for name in ('gamma','gae_lambda','actor_lr','critic_lr','clip_ratio','value_clip',
                     'entropy_coef','target_kl','grad_clip'):
            v=getattr(self,name)
            require(type(v) in (float,int) and math.isfinite(v),name+'必须有限数值')
        require(0<self.gamma<1 and 0<=self.gae_lambda<=1,'gamma或lambda非法')
        require(self.value_method in {'clipmax','plain'} and type(self.bootstrap_truncation) is bool
                and type(self.kl_stop_enabled) is bool, '消融字段类型或方法不合法')
        require(self.actor_lr>0 and self.critic_lr>0 and 0<self.clip_ratio<1
                and self.value_clip>0 and self.entropy_coef>=0 and self.target_kl>0
                and self.grad_clip>0,'训练/裁剪参数非法')

In [2]:
torch.set_num_threads(1)
seed_all(6501)
OUTPUT_ROOT = Path("lesson_65_outputs")
RUN_FULL_EXPERIMENT = True
print("Python:",platform.python_version(),"PyTorch:",torch.__version__)
print("设备：CPU双精度；CUDA本课没有实测")
print("本次运行不使用外部网络或真实账户。")

Python: 3.13.5 PyTorch: 2.10.0+cpu
设备：CPU双精度；CUDA本课没有实测
本次运行不使用外部网络或真实账户。


## 1．三状态环境：终止与截断不一样

从位置0开始，动作0等待、动作1前进，达到位置3成功并结束；超过固定 horizon 则由时间上限截断。训练奖励为每步 −0.01，加上成功时+1。

**真实终止**不再 bootstrap；**外部时间截断**仍应使用 `V(final_obs)`。把二者都当终止会改变 GAE 和 Critic 训练目标。[GAE 原论文](https://arxiv.org/abs/1506.02438)、[Gymnasium Time Limits](https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/)

本课比较两个 horizon（5、7），它们仍属于同一个小型环境，不是两类无关 benchmark。

### 实现单元：环境动态、终止、截断和 GAE

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [3]:
GOAL=3
ACTION_WAIT, ACTION_ADVANCE=0,1
STEP_COST,SUCCESS_BONUS=-.01,1.0


@dataclass(frozen=True)
class Step:
    obs: int
    action: int
    reward: float
    next_obs: int
    terminated: bool
    truncated: bool
    old_logp: float

def validate_step(step):
    require(isinstance(step, Step), "Step类型错误")
    require(type(step.obs) is int and 0 <= step.obs < GOAL, "当前obs必须是非终止位置")
    require(type(step.next_obs) is int and 0 <= step.next_obs <= GOAL, "next_obs无效")
    require(type(step.action) is int and step.action in (ACTION_WAIT, ACTION_ADVANCE), "动作非法")
    require(type(step.terminated) is bool and type(step.truncated) is bool,
            "terminated/truncated必须是bool")
    require(not (step.terminated and step.truncated), "真实终止与外部截断在本课程中互斥")
    require(step.next_obs == min(step.obs + step.action, GOAL), "环境动态不匹配")
    require(step.terminated == (step.next_obs == GOAL), "终止状态不匹配")
    expected_reward = STEP_COST + (SUCCESS_BONUS if step.terminated else 0)
    require(math.isfinite(step.reward) and abs(step.reward - expected_reward) < 1e-10,
            "奖励必须来自固定环境，而不是模型自报")
    require(type(step.old_logp) is float and math.isfinite(step.old_logp) and step.old_logp <= 1e-6,
            "需要有效的行为log概率")

def validate_episode(episode, horizon):
    require(type(episode) is list and 1 <= len(episode) <= horizon, "episode长度错误")
    require(episode[0].obs == 0, "每回合必须从位置0开始")
    for i, step in enumerate(episode):
        validate_step(step)
        if i < len(episode) - 1:
            require(not step.terminated and not step.truncated, "回合结束后仍有步骤")
            require(episode[i + 1].obs == step.next_obs, "下一步观察不连续")
    require(episode[-1].terminated or episode[-1].truncated, "回合尾部必须有明确终止状态")
    require(not episode[-1].truncated or len(episode) == horizon,
            "外部时间上限只能在约定horizon触发")

def transition(pos, action, t, horizon):
    require(type(pos) is int and 0 <= pos < GOAL, "pos非法")
    require(type(action) is int and action in (0, 1), "action非法")
    require(type(t) is int and 0 <= t < horizon, "时间步非法")
    nxt = min(pos + action, GOAL)
    terminated = nxt == GOAL
    truncated = t + 1 >= horizon and not terminated
    reward = STEP_COST + (SUCCESS_BONUS if terminated else 0.)
    return nxt, reward, terminated, truncated

def checked_numeric(values, name):
    require(isinstance(values, (list, tuple)) and len(values) > 0, name + "必须为非空序列")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), name + "必须有限")

def discounted_returns(rewards, gamma, bootstrap=0.0):
    """时间倒序累计，最后一个非终止cut使用V(final_obs)作为尾部。"""
    checked_numeric(rewards, "rewards")
    require(type(gamma) in (int, float) and 0 < gamma <= 1, "gamma非法")
    require(type(bootstrap) in (int, float) and math.isfinite(bootstrap), "bootstrap非法")
    acc = float(bootstrap)
    output = [0.] * len(rewards)
    for i in range(len(rewards)-1, -1, -1):
        acc = float(rewards[i]) + gamma*acc
        output[i] = acc
    return output

def generalized_advantage_estimation(episodes, values, next_values, gamma, lam):
    """独立episode：delta=r+gamma*(1-terminated)V(next)-V(s)。

    截断时delta保留V(final_obs)，但由于下一条记录是新回合，trace递推停止。
    自然rollout数组末尾也停止trace，若未终止则必须保留下一状态bootstrap。
    返回 A 与 lambda-return=A+V_old，不连接critic梯度。
    """
    require(type(episodes) is list and bool(episodes), "需要回合列表")
    require(type(gamma) in (float, int) and 0 < gamma <= 1, "gamma非法")
    require(type(lam) in (float, int) and 0 <= lam <= 1, "lambda非法")
    count = sum(len(ep) for ep in episodes)
    checked_numeric(values, "values")
    checked_numeric(next_values, "next_values")
    require(len(values) == len(next_values) == count, "值函数数组长度与步骤数量不符")
    advantages, targets, deltas = [], [], []
    offset = 0
    for ep in episodes:
        require(len(ep) > 0, "空回合")
        for j, step in enumerate(ep):
            validate_step(step)
            if j < len(ep)-1:
                require(not (step.terminated or step.truncated), "episode结束后仍有下一条步骤")
                require(ep[j+1].obs == step.next_obs, "GAE回合内部不连续")
        local_values = values[offset:offset + len(ep)]
        local_next = next_values[offset:offset + len(ep)]
        offset += len(ep)
        local_d = [step.reward + gamma*(0. if step.terminated else nv) - v
                   for step,v,nv in zip(ep,local_values,local_next)]
        local_adv = [0.] * len(ep)
        ahead = 0.
        for j in range(len(ep)-1, -1, -1):
            # 回合结束或采样切断后，不能把下一回合的优势连接过来。
            boundary = ep[j].terminated or ep[j].truncated
            ahead = local_d[j] + gamma*lam*(0. if boundary else ahead)
            local_adv[j] = ahead
        deltas.extend(local_d)
        advantages.extend(local_adv)
        targets.extend([a+v for a,v in zip(local_adv,local_values)])
    return {"delta":deltas, "advantage":advantages, "lambda_target":targets}

def exact_policy_values(forward_probabilities, gamma=0.95):
    """Bellman: 在底层无限时域、位置马尔可夫过程里计算精确V^pi。

    外部6步截断不属于底层MDP，所以不能把它写进该值函数。
    """
    checked_numeric(forward_probabilities, "forward_probabilities")
    require(len(forward_probabilities) == GOAL, "每个非终止位置一个前进概率")
    require(all(0 <= v <= 1 for v in forward_probabilities), "动作概率不在0..1")
    require(type(gamma) in (int,float) and 0 < gamma < 1, "无穷时域要求gamma小于1")
    values = [0.] * GOAL
    for s in range(GOAL-1, -1, -1):
        p = float(forward_probabilities[s])
        forward_value = SUCCESS_BONUS + STEP_COST if s == GOAL-1 else STEP_COST + gamma*values[s+1]
        values[s] = (p*forward_value + (1-p)*STEP_COST)/(1 - gamma*(1-p))
    return values

def finite_horizon_success_probability(forward_probabilities, horizon):
    """固定外部horizon内到达目标的精确概率，不等于底层无限时域V。"""
    require(type(horizon) is int and horizon >= 1, "horizon非法")
    checked_numeric(forward_probabilities, "policy")
    require(len(forward_probabilities)==GOAL and all(0 <= x <= 1 for x in forward_probabilities),
            "策略概率非法")
    active = [1., 0., 0.]
    finished = 0.
    for _ in range(horizon):
        new = [0.] * GOAL
        for s,pop in enumerate(active):
            prob = forward_probabilities[s]
            new[s] += pop*(1-prob)
            if s == GOAL-1:
                finished += pop*prob
            else:
                new[s+1] += pop*prob
        active = new
    require(abs(finished+sum(active)-1)<1e-9, "总概率不归一")
    return finished

In [4]:
episode=[[Step(0,0,STEP_COST,0,False,True,float(math.log(.5)))]]
correct=generalized_advantage_estimation(episode,[.2],[.8],.95,.95)
wrong=generalized_advantage_estimation(episode,[.2],[0.],.95,.95)
print("正确截断 delta:",correct['delta'][0])
print("错误丢失 bootstrap delta:",wrong['delta'][0])
print("差额=gamma*V(final_obs):",correct['delta'][0]-wrong['delta'][0])

正确截断 delta: 0.55
错误丢失 bootstrap delta: -0.21000000000000002
差额=gamma*V(final_obs): 0.76


### GAE λ 不是“是否使用 GAE”的简单开关

在同一采样数据和旧价值预测下，`λ=0` 等价于单步 TD residual；`λ` 接近1会累积更长的 TD residual。二者带来不同偏差/方差与 critic target，也可能改变整个之后的采样轨迹。

因此本课登记的消融叫 **GAE λ：0.95→0**，不将 `λ=0` 草率地命名为“没有任何优势估计”。训练时仍旧使用合法的 advantage 与 bootstrap。

为训练两个 horizon 的比较，基座始终采用完全相同的价值/训练定义；不同设置以显式配置变化登记。

## 2．Actor/Critic 与留出主要指标

基座是独立 Actor 和 Critic。Actor 在三个状态输出等待或前进的 logit；Critic 对每个非终止状态估计价值。

主要任务指标是**外部 horizon 内达到目标的精确成功概率**，由有限步动态规划计算，不依赖有限采样的成功回合数。另记录随机采样成功数和 Critic 在当前策略**无限视野**价值上的 RMSE：这两者不是开发选择用的主要指标。

小环境能计算精确主指标，是为了排除评估 Monte Carlo 噪声；它不证明 PPO 训练本身没有采样噪声。

### 实现单元：Actor/Critic、采样与精确评价

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [5]:
class Actor(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,2))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "actor状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "actor状态越界")
        return self.net(states)

class Critic(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,1))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "critic状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "critic状态越界")
        return self.net(states).squeeze(-1)

def actor_probabilities(actor):
    with torch.no_grad():
        actor.eval()
        probs=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)[:,ACTION_ADVANCE]
    return probs.tolist()

def value_prediction(critic):
    with torch.no_grad():
        critic.eval()
        return critic(torch.arange(GOAL,dtype=torch.long)).tolist()

def value_error(critic,actor,gamma):
    reference=exact_policy_values(actor_probabilities(actor),gamma)
    prediction=value_prediction(critic)
    return {"predicted":prediction,"exact_for_current_policy":reference,
            "rmse":math.sqrt(sum((a-b)**2 for a,b in zip(prediction,reference))/GOAL),
            "scope":"exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return"}

def collect_rollouts(actor, episodes, horizon, rng):
    require(type(episodes) is int and episodes >= 1, "episodes非法")
    require(type(horizon) is int and horizon >= 3, "horizon非法")
    result=[]
    actor.eval()
    with torch.no_grad():
        p=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)
    for _ in range(episodes):
        pos=0;ep=[]
        for t in range(horizon):
            action=1 if rng.random()<float(p[pos,1]) else 0
            logp=math.log(float(p[pos,action]))
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,logp))
            if terminated or truncated:
                break
            pos=nxt
        validate_episode(ep,horizon)
        result.append(ep)
    return result

def compute_targets(episodes,critic,gamma,lam,bootstrap_truncation=True):
    require(type(bootstrap_truncation) is bool, "bootstrap标志必须为bool")
    states=[s.obs for ep in episodes for s in ep]
    nextstates=[s.next_obs if not s.terminated else 0 for ep in episodes for s in ep]
    with torch.no_grad():
        values=critic(torch.tensor(states,dtype=torch.long)).tolist()
        nextvalues=critic(torch.tensor(nextstates,dtype=torch.long)).tolist()
    for i,step in enumerate(s for ep in episodes for s in ep):
        if step.terminated or (step.truncated and not bootstrap_truncation):
            nextvalues[i]=0.
    target=generalized_advantage_estimation(episodes,values,nextvalues,gamma,lam)
    return {"states":states,"next_states":nextstates,"values_old":values,
            "next_values_old":nextvalues,"advantages":target['advantage'],
            "lambda_returns":target['lambda_target'],"deltas":target['delta'],
            "old_logps":[s.old_logp for ep in episodes for s in ep],
            "actions":[s.action for ep in episodes for s in ep],
            "rewards":[s.reward for ep in episodes for s in ep],
            "terminated":[s.terminated for ep in episodes for s in ep],
            "truncated":[s.truncated for ep in episodes for s in ep],
            "episodes":len(episodes),"steps":len(states),
            "bootstrap_truncation":bootstrap_truncation,
            "scope":"on_policy_fixed_targets_detached_from_critic"}

def evaluate_policy(actor,critic,gamma,horizon,rollouts=200,seed=2026):
    prob=actor_probabilities(actor)
    exact_continuing=exact_policy_values(prob,gamma)
    exact_success=finite_horizon_success_probability(prob,horizon)
    sampled=collect_rollouts(actor,rollouts,horizon,random.Random(seed))
    success=sum(ep[-1].terminated for ep in sampled)
    truncated=sum(ep[-1].truncated for ep in sampled)
    return {"forward_probabilities":prob,"exact_continuing_value":exact_continuing,
            "exact_success_within_limit":exact_success,
            "sampled_successes":success,"sampled_episodes":rollouts,
            "sampled_success_rate":success/rollouts,"sampled_truncations":truncated,
            "value_rmse":value_error(critic,actor,gamma)['rmse'],
            "note":"固定环境的精确计算＋有限采样；并非模型数学推理能力"}

In [6]:
seed_all(6501)
actor_demo=Actor(24).double();critic_demo=Critic(24).double()
print("初始Actor前进概率:",actor_probabilities(actor_demo))
print("固定horizon=5下成功概率:",finite_horizon_success_probability(actor_probabilities(actor_demo),5))
print("Critic RMSE诊断:",value_error(critic_demo,actor_demo,.95)['rmse'])

初始Actor前进概率: [0.4247624312835734, 0.5221234345403012, 0.41713126998400996]
固定horizon=5下成功概率: 0.40923991692542777
Critic RMSE诊断: 0.6577871190322997


## 3．Value loss 的三个名字不能混用

| 本课名称 | 明确定义 |
|---|---|
| `plain` | `0.5 * mean((V-R)²)` |
| `clipmax` | `0.5 * mean(max((V-R)²,(V_clip-R)²))` |
| `clipped_pred` | `0.5 * mean((V_clip-R)²)` |

其中 `V_clip=V_old+clamp(V-V_old,-eps,eps)`。

后两种是**不同**的可选工程变体，并不都等于原 PPO 论文唯一指定的值损失。Stable-Baselines3 提供过 `clipped_pred` 一类可选方案；PPO2 风格还有 `clipmax`。如果两篇报告都写“value clipping”，却没有给公式，就无法归因。[SB3 PPO代码](https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html)、[Implementation Matters](https://arxiv.org/abs/2005.12729)

本课**主消融**使用 `clipmax` 与 `plain`。`clipped_pred` 只用于机制反例，不偷换为主消融。数值实验会同时核对值与梯度。

### 实现单元：PPO actor裁剪、不同value loss及KL

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [7]:
def probability_ratio(new_logp, old_logp):
    require(isinstance(new_logp,torch.Tensor) and isinstance(old_logp,torch.Tensor)
            and new_logp.shape==old_logp.shape and new_logp.ndim==1 and new_logp.numel()>0,
            '新旧log概率需要同形状非空向量')
    require(new_logp.is_floating_point() and old_logp.is_floating_point()
            and new_logp.dtype==old_logp.dtype and new_logp.device==old_logp.device,
            '新旧log概率精度/设备非法')
    require(bool(torch.isfinite(new_logp).all() and torch.isfinite(old_logp).all()),'log概率非有限')
    delta=new_logp-old_logp.detach()
    require(bool((delta.detach().abs()<40).all()),'概率比过大，不静默截断')
    return delta.exp()

def policy_loss(new_logp, old_logp, advantage, epsilon):
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0<epsilon<1,'policy clip范围非法')
    require(isinstance(advantage,torch.Tensor) and advantage.shape==new_logp.shape
            and advantage.dtype==new_logp.dtype and advantage.device==new_logp.device
            and bool(torch.isfinite(advantage).all()),'优势形状/精度/数值非法')
    ratio=probability_ratio(new_logp,old_logp)
    a=advantage.detach()
    plain=ratio*a
    bounded=ratio.clamp(1-epsilon,1+epsilon)*a
    obj=torch.minimum(plain,bounded)
    outside=(ratio>1+epsilon)|(ratio<1-epsilon)
    active=((ratio>1+epsilon)&(a>0))|((ratio<1-epsilon)&(a<0))
    return -obj.mean(),{'ratio':ratio,'surrogate':obj,'plain':plain,
                        'ratio_outside_fraction':float(outside.detach().float().mean()),
                        'active_clip_fraction':float(active.detach().float().mean())}

def value_loss(prediction, old_value, target, epsilon=.2, method='clipmax'):
    """三种可选值损失；clipmax不是原始PPO论文的必选部件。

    plain:        0.5*MSE(pred,target)
    clipmax:      0.5*mean(max(raw_squared, clipped_pred_squared)) (PPO2变体)
    clipped_pred: 0.5*MSE(clipped_prediction,target) (SB3可选实现)
    """
    require(method in {'plain','clipmax','clipped_pred'},'value loss方法不支持')
    require(type(epsilon) in (float,int) and math.isfinite(epsilon) and epsilon>0,'value epsilon无效')
    require(all(isinstance(x,torch.Tensor) for x in (prediction,old_value,target))
            and prediction.ndim==1 and prediction.numel()>0
            and prediction.shape==old_value.shape==target.shape,'value输入必须同形状一维向量')
    require(all(x.is_floating_point() and x.dtype==prediction.dtype and x.device==prediction.device
                and bool(torch.isfinite(x).all()) for x in (prediction,old_value,target)),
            'value设备、dtype或数值非法')
    old=old_value.detach(); goal=target.detach()
    raw_squared=(prediction-goal).square()
    clipped_prediction=old+(prediction-old).clamp(-epsilon,epsilon)
    clipped_squared=(clipped_prediction-goal).square()
    if method=='plain': squared=raw_squared
    elif method=='clipmax': squared=torch.maximum(raw_squared,clipped_squared)
    else: squared=clipped_squared
    return .5*squared.mean(), {
        'raw_half_mse':float((.5*raw_squared.detach()).mean()),
        'clipped_prediction_half_mse':float((.5*clipped_squared.detach()).mean()),
        'max_half_mse':float((.5*torch.maximum(raw_squared.detach(),clipped_squared.detach())).mean()),
        'value_outside_clip_fraction':float(((prediction.detach()-old).abs()>epsilon).float().mean()),
        'method':method}

def normalize_advantages(advantages):
    require(isinstance(advantages,torch.Tensor) and advantages.ndim==1 and advantages.numel()>1
            and advantages.is_floating_point() and bool(torch.isfinite(advantages).all()),
            '归一化优势需要超过一个有限浮点样本')
    a=advantages.detach()
    return (a-a.mean())/(a.std(unbiased=False)+1e-8)

def old_state_kl(old_distribution,new_distribution):
    """仅计算旧样本访问的三状态上的Categorical KL(old||new)的平均。"""
    require(isinstance(old_distribution,torch.Tensor) and isinstance(new_distribution,torch.Tensor)
            and old_distribution.ndim==2 and old_distribution.shape==new_distribution.shape
            and old_distribution.shape[1]==2 and old_distribution.numel()>0,'KL分布形状无效')
    require(all(torch.isfinite(x).all() for x in (old_distribution,new_distribution)),'非有限分布')
    for d in (old_distribution,new_distribution):
        require(bool(torch.allclose(d.logsumexp(-1),torch.zeros_like(d[:,0]),atol=1e-5)),
                'KL需要归一化log概率')
    old=old_distribution.detach()
    return (old.exp()*(old-new_distribution)).sum(-1).mean()

In [8]:
pred=torch.tensor([3.,1.],dtype=torch.float64,requires_grad=True)
oldv=torch.tensor([0.,0.],dtype=torch.float64)
returns=torch.tensor([2.,0.],dtype=torch.float64)
for method in ['plain','clipmax','clipped_pred']:
    vloss,_=value_loss(pred,oldv,returns,epsilon=.2,method=method)
    print(method, 'loss=',float(vloss.detach()),
          'gradient=',torch.autograd.grad(vloss,pred,retain_graph=True)[0].tolist())

plain loss= 0.5 gradient= [0.5, 0.5]
clipmax loss= 1.06 gradient= [0.0, 0.5]
clipped_pred loss= 0.8200000000000001 gradient= [0.0, 0.0]


## 4．冻结rollout与一次性优势归一化

每次采样产生变长episode，之后统一计算 GAE 与固定 `lambda_return`。在整个 rollout 上一次性归一化优势，再做多轮 minibatch 复用；不会每个 minibatch 重新归一化，也不把 old value 刷新成当前预测。

保留封存摘要：states、actions、old_logps、old_values、target、advantage、old_distribution，以及终止/截断标签都不能在优化中偷偷变化。

`actor_stopped` 只禁止后续 Actor 更新；Critic 仍可能继续优化当前 rollout 的旧目标。是否做这种设计本身也要在报告中说明。

### 实现单元：rollout封存与增量审计

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [9]:
def stable_epoch_seed(master: int, rollout_id: int, epoch: int) -> int:
    require(all(type(x) is int and x>=0 for x in (master, rollout_id, epoch)),"种子与索引必须非负整数")
    return int(sha256(["lesson63", master, rollout_id, epoch])[:8],16)


def shuffled_minibatches(step_count: int, minibatch_size: int, seed: int) -> list[list[int]]:
    require(type(step_count) is int and step_count>0,"没有rollout步骤")
    require(type(minibatch_size) is int and minibatch_size>0,"minibatch size非法")
    rng=random.Random(seed)
    ids=list(range(step_count))
    rng.shuffle(ids)
    return [ids[start:start+minibatch_size] for start in range(0,step_count,minibatch_size)]


def split_microbatches(minibatch: list[int], microbatch_size: int) -> list[list[int]]:
    require(type(microbatch_size) is int and microbatch_size>0,"microbatch size非法")
    require(isinstance(minibatch,list) and bool(minibatch),"minibatch为空")
    require(len(set(minibatch))==len(minibatch),"minibatch有重复索引")
    return [minibatch[i:i+microbatch_size] for i in range(0,len(minibatch),microbatch_size)]


def frozen_signature(buffer: dict) -> str:
    names=("states","actions","old_logps","old_values","returns","advantages",
           "raw_advantages","old_distribution","terminated","truncated")
    frozen_tensors=tensor_hash({name:buffer[name] for name in names})
    meta={k:buffer[k] for k in ("rollout_id","episode_lengths","episode_successes",
                              "episode_truncations","sampled_steps")}
    return sha256([frozen_tensors,meta])


def make_buffer(actor: Actor, critic: Critic, episodes: list[list[Step]], cfg: Config,
                rollout_id: int) -> dict:
    cfg.validate()
    require(type(rollout_id) is int and rollout_id>=0,"rollout ID非法")
    require(len(episodes)==cfg.episodes_per_rollout,"episode数量与配置不同")
    for ep in episodes:validate_episode(ep,cfg.horizon)
    values=compute_targets(episodes,critic,cfg.gamma,cfg.gae_lambda,bootstrap_truncation=cfg.bootstrap_truncation)
    states=torch.tensor(values['states'],dtype=torch.long)
    actions=torch.tensor(values['actions'],dtype=torch.long)
    old_logps=torch.tensor(values['old_logps'],dtype=torch.float64)
    old_values=torch.tensor(values['values_old'],dtype=torch.float64)
    returns=torch.tensor(values['lambda_returns'],dtype=torch.float64)
    raw=torch.tensor(values['advantages'],dtype=torch.float64)
    require(len(raw)>1,"全局优势归一化至少2步")
    normalized=(raw-raw.mean())/(raw.std(unbiased=False)+1e-8)
    with torch.no_grad():
        dist=F.log_softmax(actor(states),dim=-1)
        recalculated=dist.gather(-1,actions.unsqueeze(-1)).squeeze(-1)
    require(torch.allclose(old_logps,recalculated,atol=2e-6,rtol=2e-6),
            "采样时old_logp与当前Actor重算不一致")
    result={"states":states,"actions":actions,"old_logps":old_logps,
            "old_values":old_values,"returns":returns,
            "advantages":normalized.detach().clone(),"raw_advantages":raw,
            "old_distribution":dist.detach().clone(),
            "terminated":torch.tensor(values['terminated'],dtype=torch.bool),
            "truncated":torch.tensor(values['truncated'],dtype=torch.bool),
            "rollout_id":rollout_id,"sampled_steps":len(raw),
            "episode_lengths":[len(ep) for ep in episodes],
            "episode_successes":sum(ep[-1].terminated for ep in episodes),
            "episode_truncations":sum(ep[-1].truncated for ep in episodes),
            "actor_stopped":False,"closed":False,"finished_epochs":0,
            "actor_updates":0,"critic_updates":0,
            "actor_before_hash":tensor_hash(actor.state_dict()),
            "critic_before_hash":tensor_hash(critic.state_dict()),
            "expected_actor_hash":tensor_hash(actor.state_dict()),
            "expected_critic_hash":tensor_hash(critic.state_dict())}
    result['seal']=frozen_signature(result)
    return result


def verify_buffer(actor: Actor, critic: Critic,buffer: dict,cfg: Config,rollout_id: int):
    require(not buffer['closed'],"rollout已经关闭")
    require(buffer['rollout_id']==rollout_id,"不能跨rollout使用旧数据")
    require(buffer['seal']==frozen_signature(buffer),"冻结的rollout数组或元数据被篡改")
    require(buffer['finished_epochs']<cfg.epochs,"已超过计划epoch数")
    require(buffer['expected_actor_hash']==tensor_hash(actor.state_dict()),"Actor在受控循环外变化")
    require(buffer['expected_critic_hash']==tensor_hash(critic.state_dict()),"Critic在受控循环外变化")
    require(all(not buffer[n].requires_grad for n in (
        'old_logps','old_values','returns','advantages','old_distribution')),
        "旧策略、GAE与value target不得持有梯度图")

In [10]:
cfg_demo=Config(seed=6501,horizon=5,rollouts=2,episodes_per_rollout=9,
                epochs=2,minibatch_size=17,microbatch_size=5)
seed_all(6501)
actor_demo=Actor(24).double();critic_demo=Critic(24).double()
episodes=collect_rollouts(actor_demo,9,5,random.Random(6503))
buffer=make_buffer(actor_demo,critic_demo,episodes,cfg_demo,0)
print('episode数:',len(episodes),'有效transition:',buffer['sampled_steps'])
print('终止:',buffer['episode_successes'],'截断:',buffer['episode_truncations'])
print('rollout seal长度:',len(buffer['seal']))

episode数: 9 有效transition: 44
终止: 3 截断: 6
rollout seal长度: 64


## 5．Batching 效应与微批等价性：两个不同问题

本课有三种模式：

- `accum`：有效 mini 为17 transition，但每次前向只处理最多5 transition，梯度按17步归一。
- `mini`：同一有效17 transition，直接一次前向反传；这是**实现等价性测试**，不是训练算法因素。
- `full`：把整个 rollout 作为一个更新 mini，每个 epoch 只执行一个 optimizer step。它改变了**更新次数和批次宽度**，不能叫“只隔离硬件microbatch大小”。

尾部短 microbatch 必须按真实行数累计，不能与5步 microbatch 等权。数值等价性与 GPU 显存/吞吐是不同问题；本课 CPU 双精度只验证前者。

### 实现单元：真实 mini/micro loss、梯度累积与KL门控

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [11]:
def per_step_losses(actor: Actor,critic: Critic,buffer: dict,index: torch.Tensor,cfg: Config):
    require(index.dtype==torch.long and index.ndim==1 and index.numel()>0,"需要非空long索引")
    s=buffer['states'][index];a=buffer['actions'][index]
    old_lp=buffer['old_logps'][index]
    adv=buffer['advantages'][index].detach()
    old_v=buffer['old_values'][index].detach()
    target=buffer['returns'][index].detach()
    dist=F.log_softmax(actor(s),dim=-1)
    current_lp=dist.gather(-1,a.unsqueeze(-1)).squeeze(-1)
    ratio=probability_ratio(current_lp,old_lp)
    plain=ratio*adv
    clipped=ratio.clamp(1-cfg.clip_ratio,1+cfg.clip_ratio)*adv
    entropy=-(dist.exp()*dist).sum(-1)
    actor_per=-torch.minimum(plain,clipped)-cfg.entropy_coef*entropy
    predicted=critic(s)
    raw_square=(predicted-target).square()
    clipped_values=old_v+(predicted-old_v).clamp(-cfg.value_clip,cfg.value_clip)
    if cfg.value_method == 'clipmax':
        critic_per=.5*torch.maximum(raw_square,(clipped_values-target).square())
    else:
        critic_per=.5*raw_square
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    outside=(ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio)
    return actor_per,critic_per,{
        'ratio':ratio.detach(),'active_clip':active.detach(),'outside':outside.detach(),
        'entropy':entropy.detach(), 'old_minus_new':(old_lp-current_lp).detach(),
        'value_abs_error':(predicted-target).abs().detach()}


@torch.no_grad()
def rollout_diagnostics(actor: Actor,critic: Critic,buffer: dict,cfg: Config) -> dict:
    """整个已观察rollout按step频数统计，而不是平均3个状态或minibatch均值。"""
    actor.eval();critic.eval()
    dist=F.log_softmax(actor(buffer['states']),dim=-1)
    old=buffer['old_distribution'];a=buffer['actions']
    current_lp=dist.gather(-1,a[:,None]).squeeze(-1)
    ratio=probability_ratio(current_lp,buffer['old_logps'])
    old_minus_new=buffer['old_logps']-current_lp
    pointwise=(old.exp()*(old-dist)).sum(-1)
    require(float(pointwise.min())>-1e-8,"归一化分类KL数值异常")
    sample_estimator=((ratio-1)-ratio.log()).mean()  # 非负的近似估计形式
    adv=buffer['advantages']
    out=((ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio))
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    ent=-(dist.exp()*dist).sum(-1)
    predicted=critic(buffer['states'])
    result={'transitions':len(a),
        'old_state_kl_exact_visited':float(pointwise.mean()),
        'sampled_approx_kl':float(old_minus_new.mean()),
        'sampled_nonnegative_estimator':float(sample_estimator),
        'sampled_approx_may_be_negative':True,
        'ratio_min':float(ratio.min()), 'ratio_max':float(ratio.max()),
        'ratio_outside_fraction':float(out.double().mean()),
        'active_clip_fraction':float(active.double().mean()),
        'entropy_per_transition':float(ent.mean()),
        'critic_half_mse':float((.5*(predicted-buffer['returns']).square()).mean()),
        'mean_normalized_advantage':float(adv.mean()),
        'old_state_frequency':[int((buffer['states']==s).sum()) for s in range(GOAL)]}
    return result


def full_minibatch_gradients(actor: Actor,critic: Critic,buffer: dict,
                             minibatch: list[int],cfg: Config, microbatch_size: int,
                             compute_actor: bool=True):
    """通过一个采样minibatch中的microchunks累积；此函数不执行step。

    每个micro的loss按该micro行数求和后 / 当前minibatch总行数。
    对actor/critic分别反向，保证最后不满micro时也等价于直接full-minibatch求均值。
    """
    require(type(microbatch_size) is int and microbatch_size>0,'micro大小非法')
    require(bool(minibatch) and len(set(minibatch))==len(minibatch),"minibatch索引重复/空")
    require(all(type(i) is int and 0<=i<buffer['sampled_steps'] for i in minibatch),"索引越界")
    actor.zero_grad(set_to_none=True);critic.zero_grad(set_to_none=True)
    mini_size=len(minibatch)
    parts=split_microbatches(minibatch,microbatch_size)
    sums={'actor_loss_sum':0.,'critic_loss_sum':0.,'entropy_sum':0.,'clip_count':0,
          'outside_count':0,'old_minus_new_sum':0.,'abs_value_error_sum':0.}
    for micro in parts:
        index=torch.tensor(micro,dtype=torch.long)
        actor_per,critic_per,trace=per_step_losses(actor,critic,buffer,index,cfg)
        # Do NOT divide by len(parts) or by len(micro) here.
        if compute_actor:
            (actor_per.sum()/mini_size).backward()
        (critic_per.sum()/mini_size).backward()
        with torch.no_grad():
            sums['actor_loss_sum']+=float(actor_per.detach().sum())
            sums['critic_loss_sum']+=float(critic_per.detach().sum())
            sums['entropy_sum']+=float(trace['entropy'].sum())
            sums['clip_count']+=int(trace['active_clip'].sum())
            sums['outside_count']+=int(trace['outside'].sum())
            sums['old_minus_new_sum']+=float(trace['old_minus_new'].sum())
            sums['abs_value_error_sum']+=float(trace['value_abs_error'].sum())
    sums.update({'size':mini_size,'micro_sizes':[len(p) for p in parts],
                 'actor_loss_before':sums['actor_loss_sum']/mini_size,
                 'critic_loss_before':sums['critic_loss_sum']/mini_size,
                 'entropy_before':sums['entropy_sum']/mini_size,
                 'active_clip_fraction_before':sums['clip_count']/mini_size,
                 'ratio_outside_fraction_before':sums['outside_count']/mini_size,
                 'sampled_approx_kl_before':sums['old_minus_new_sum']/mini_size})
    return sums


def minibatch_update(actor: Actor,critic: Critic,buffer: dict,minibatch: list[int],
                     cfg: Config,actor_optimizer,critic_optimizer,microbatch_size: int) -> dict:
    require(bool(minibatch),"minibatch为空")
    old_diag=rollout_diagnostics(actor,critic,buffer,cfg)
    if cfg.kl_stop_enabled and not buffer['actor_stopped'] and old_diag['old_state_kl_exact_visited']>cfg.target_kl:
        buffer['actor_stopped']=True
    actor_update=not buffer['actor_stopped']
    actor_opt={id(p) for g in actor_optimizer.param_groups for p in g['params']}
    critic_opt={id(p) for g in critic_optimizer.param_groups for p in g['params']}
    require(actor_opt=={id(p) for p in actor.parameters()},"Actor优化器对象错误")
    require(critic_opt=={id(p) for p in critic.parameters()},"Critic优化器对象错误")
    require(not actor_opt&critic_opt,"Actor/Critic优化器不能共享对象")
    stats=full_minibatch_gradients(actor,critic,buffer,minibatch,cfg,microbatch_size,
                                   compute_actor=actor_update)
    if not actor_update:
        actor.zero_grad(set_to_none=True)
        actor_norm=None
    else:
        actor_norm=float(nn.utils.clip_grad_norm_(actor.parameters(),cfg.grad_clip,error_if_nonfinite=True))
        actor_optimizer.step()
        buffer['actor_updates']+=1
    critic_norm=float(nn.utils.clip_grad_norm_(critic.parameters(),cfg.grad_clip,error_if_nonfinite=True))
    critic_optimizer.step();buffer['critic_updates']+=1
    buffer['expected_actor_hash']=tensor_hash(actor.state_dict())
    buffer['expected_critic_hash']=tensor_hash(critic.state_dict())
    require(buffer['seal']==frozen_signature(buffer),"优化期间旧采样或目标被改变")
    return {'size':stats['size'],'micro_sizes':stats['micro_sizes'],
            'actor_updated':actor_update,'critic_updated':True,
            'actor_stopped':buffer['actor_stopped'],
            'old_state_kl_at_gate':old_diag['old_state_kl_exact_visited'],
            'sampled_kl_at_gate':old_diag['sampled_approx_kl'],
            'actor_grad_norm_before_clip':actor_norm,'critic_grad_norm_before_clip':critic_norm,
            **{k:v for k,v in stats.items() if k in ('actor_loss_before','critic_loss_before',
                'entropy_before','active_clip_fraction_before','ratio_outside_fraction_before',
                'sampled_approx_kl_before','clip_count','outside_count','actor_loss_sum',
                'critic_loss_sum','entropy_sum','old_minus_new_sum')}}

In [12]:
order=shuffled_minibatches(buffer['sampled_steps'],17,stable_epoch_seed(6501,0,0))
first=order[0]
full_grad=full_minibatch_gradients(actor_demo,critic_demo,buffer,first,cfg_demo,17)
actor_expected=[p.grad.clone() for p in actor_demo.parameters()]
critic_expected=[p.grad.clone() for p in critic_demo.parameters()]
micro_grad=full_minibatch_gradients(actor_demo,critic_demo,buffer,first,cfg_demo,5)
print('初始旧状态KL:',rollout_diagnostics(actor_demo,critic_demo,buffer,cfg_demo)['old_state_kl_exact_visited'])
print('有效minibatch:',len(first),'micro形状:',micro_grad['micro_sizes'])
print('Actor梯度一致:',all(torch.allclose(x,y.grad,atol=1e-10,rtol=1e-10)
                             for x,y in zip(actor_expected,actor_demo.parameters())))
print('Critic梯度一致:',all(torch.allclose(x,y.grad,atol=1e-10,rtol=1e-10)
                             for x,y in zip(critic_expected,critic_demo.parameters())))

初始旧状态KL: 0.0
有效minibatch: 17 micro形状: [5, 5, 5, 2]
Actor梯度一致: True
Critic梯度一致: True


## 6．KL stop 的准确含义

本课每个当前 minibatch 更新前，评估旧 rollout 已访问状态上的平均分类 KL（old‖current）。若 `kl_stop_enabled=True` 且超过预设阈值，则禁止后续 Actor step；不回滚已经提交的参数变化。

它不是全状态硬约束。分布诊断不等于有限样本 `old_logp-new_logp` 的均值；后者可负。Critic 停止与否也必须单独记录。

主消融是只关闭 `kl_stop_enabled`。不能同时改 `target_kl`、Actor 学习率和 epoch 数，再把差异都归因于“没有KL”。

## 7．训练器：冻结调度与每轮真实预算

沿用验证过的 PPO 训练循环：独立初始化、相同 seed、相同 rollout seed 调度。各消融只有登记字段不同，但随着梯度改变，后面的轨迹会分叉；**共同随机种子不等于所有条件使用相同状态动作路径**。

日志同时保存：新环境步数、重复呈现次数、Actor step、Critic step、KL停止次数、minibatch权重和各轮结果。这些预算不应该合并成一个没有单位的“训练步数”。

### 实现单元：完整变长 rollout 与训练循环

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [13]:
def summarise_preupdate_minibatches(rows: list[dict]) -> dict:
    """同一epoch所观察的minibatch，按有效transition总数求平均；短尾不获等权。"""
    require(isinstance(rows,list) and bool(rows),"至少一个minibatch")
    total=sum(r['size'] for r in rows)
    require(total>0,"没有transition")
    keys={'actor_loss':'actor_loss_sum', 'critic_loss':'critic_loss_sum',
          'entropy':'entropy_sum', 'sampled_approx_kl':'old_minus_new_sum'}
    result={k:sum(r[v] for r in rows)/total for k,v in keys.items()}
    result.update({'observed_steps':total, 'minibatches':len(rows),
                   'active_clip_fraction':sum(r['clip_count'] for r in rows)/total,
                   'ratio_outside_fraction':sum(r['outside_count'] for r in rows)/total,
                   'actor_updates':sum(r['actor_updated'] for r in rows),
                   'critic_updates':sum(r['critic_updated'] for r in rows),
                   'short_tail_size':rows[-1]['size'],
                   'note':'minibatch BEFORE-update metrics, weighted by actual transitions; not one simultaneous full-buffer evaluation'})
    return result


def train_arm(cfg: Config,mode: str,schedule: list[int],
              original_actor: Actor,original_critic: Critic):
    require(mode in {'full','mini','accum'},"未知训练对照")
    actor=copy.deepcopy(original_actor);critic=copy.deepcopy(original_critic)
    aopt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    vopt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    history=[];a_updates=c_updates=seen=total_episodes=stopped_rollouts=0
    for rid,seed in enumerate(schedule):
        eps=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(seed))
        buf=make_buffer(actor,critic,eps,cfg,rid)
        n=buf['sampled_steps']
        require(len(set(buf['episode_lengths']))>1 or cfg.episodes_per_rollout==1 or n>0,
                "回合长度异常")
        start_diag=rollout_diagnostics(actor,critic,buf,cfg)
        require(start_diag['old_state_kl_exact_visited']<1e-10,"新rollout的old state KL应为0")
        item={'rollout_id':rid,'episodes':cfg.episodes_per_rollout,'steps':n,
              'episode_lengths':buf['episode_lengths'],
              'terminations':buf['episode_successes'],'truncations':buf['episode_truncations'],
              'advantage_raw_mean':float(buf['raw_advantages'].mean()),
              'advantage_raw_std':float(buf['raw_advantages'].std(unbiased=False)),
              'advantages_normalized_once':True,
              'initial':start_diag,'epochs':[]}
        for epoch in range(cfg.epochs):
            mini_size=n if mode=='full' else cfg.minibatch_size
            order=shuffled_minibatches(n,mini_size,stable_epoch_seed(cfg.seed,rid,epoch))
            check_indices=[i for chunk in order for i in chunk]
            require(sorted(check_indices)==list(range(n)),"每epoch必须恰好覆盖所有transition")
            rows=[]
            for mini in order:
                micro_size=len(mini) if mode in {'full','mini'} else cfg.microbatch_size
                row=minibatch_update(actor,critic,buf,mini,cfg,aopt,vopt,micro_size)
                rows.append(row)
            aggregated=summarise_preupdate_minibatches(rows)
            require(aggregated['observed_steps']==n,"缺失transition或重复计算")
            item['epochs'].append({'epoch':epoch+1,'seed':stable_epoch_seed(cfg.seed,rid,epoch),
                                    'coverage_sha256':sha256(order),'batches':rows,
                                    'weighted':aggregated,
                                    'final_kl':rollout_diagnostics(actor,critic,buf,cfg)})
            buf['finished_epochs']+=1
            # 检查这一个epoch的后验状态仍满足被冻结的合同。
            require(buf['seal']==frozen_signature(buf),"GAE、回报或旧策略在epoch间变化")
        buf['closed']=True
        a_updates+=buf['actor_updates'];c_updates+=buf['critic_updates']
        stopped_rollouts+=int(buf['actor_stopped'])
        total_episodes+=cfg.episodes_per_rollout;seen+=n
        item['actor_updates']=buf['actor_updates'];item['critic_updates']=buf['critic_updates']
        item['actor_stopped']=buf['actor_stopped']
        if rid==0 or (rid+1)%8==0 or rid+1==len(schedule):
            dev=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,60,seed=6306)
            item['dev_observation']={'exact_success_within_limit':dev['exact_success_within_limit'],
                                     'value_rmse':dev['value_rmse']}
        history.append(item)
    return actor,critic,{'mode':mode,'history':history,'rollouts':len(schedule),
                         'episodes':total_episodes,'unique_sampled_steps':seen,
                         'actor_optimizer_steps':a_updates,'critic_optimizer_steps':c_updates,
                         'actor_stopped_rollouts':stopped_rollouts,
                         'transition_presentations':seen*cfg.epochs,
                         'selection':'fixed_final_update_no_test_selection',
                         'minibatch_rule':'full=single_step;mini/accum=17_transition_minis_with_5_transition_micros'}

## 8．预先登记消融清单

主逐因素条件：GAE λ=0、`plain` critic、关闭KL门控、整批更新。另有微批等价检查与GAE/value的**两因素组合**。

| 条件 | 主要目的 | 是否参与开发选择 |
|---|---|---|
| `baseline` | 固定基线 | 是 |
| `gae_zero` | GAE λ 0.95→0 | 是 |
| `value_plain` | clipmax→plain | 是 |
| `no_kl_stop` | 不再停Actor | 是 |
| `full_batch` | 真实改变mini宽度与更新数 | 是，但预算不等价 |
| `micro_unsplit` | 同样有效mini，拆分实现一致性 | 否 |
| `gae_value_combo` | GAE/value两因素交互描述 | 否 |

**开发集锁定选择**只使用主指标 `exact_success_within_limit`，按任务等权平均。不能把辅助的随机采样成功数或 Critic RMSE偷偷替换为主要评分。

### 实现单元：任务、候选、种子和预算合同

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [14]:
BASE_HPARAMS = {
    'gamma': .95, 'gae_lambda': .95, 'horizon': 5,
    'rollouts': 8, 'episodes_per_rollout': 12, 'epochs': 2,
    'minibatch_size': 17, 'microbatch_size': 5,
    'actor_lr': .012, 'critic_lr': .010, 'actor_hidden': 24,
    'critic_hidden': 24, 'clip_ratio': .2, 'value_clip': .2,
    'entropy_coef': .006, 'target_kl': .025, 'grad_clip': 1.,
    'eval_episodes': 70, 'value_method': 'clipmax',
    'bootstrap_truncation': True, 'kl_stop_enabled': True,
}


def ablation_catalog() -> list[dict]:
    """类别声明：单项消融、微批等价性检查、双因素交互。"""
    return [
        {'id':'baseline', 'label':'基线：GAE0.95、clipmax、KL停止、17/5累积',
         'changes':{}, 'train_mode':'accum', 'category':'baseline', 'selectable':True},
        {'id':'gae_zero', 'label':'GAE lambda: 0.95 → 0',
         'changes':{'gae_lambda':0.0}, 'train_mode':'accum', 'category':'one_factor', 'selectable':True},
        {'id':'value_plain', 'label':'Critic clipmax → plain 0.5 MSE',
         'changes':{'value_method':'plain'}, 'train_mode':'accum', 'category':'one_factor', 'selectable':True},
        {'id':'no_kl_stop', 'label':'仅关闭 Actor 的KL停止',
         'changes':{'kl_stop_enabled':False}, 'train_mode':'accum', 'category':'one_factor', 'selectable':True},
        {'id':'full_batch', 'label':'Mini累积 → 每epoch完整batch一次更新',
         'changes':{}, 'train_mode':'full', 'category':'one_factor', 'selectable':True},
        {'id':'micro_unsplit', 'label':'5-step micro累积 → 直接17-step minibatch',
         'changes':{}, 'train_mode':'mini', 'category':'implementation_equivalence', 'selectable':False},
        {'id':'gae_value_combo', 'label':'GAE0 + plain value，交互诊断',
         'changes':{'gae_lambda':0.0,'value_method':'plain'}, 'train_mode':'accum',
         'category':'two_factor_interaction', 'selectable':False},
    ]


def make_protocol(quick: bool=False) -> dict:
    require(type(quick) is bool, 'quick应为bool')
    candidates=ablation_catalog()
    return {'version':VERSION, 'variant':'quick_smoke' if quick else 'registered_ablation',
            'tasks':[{'id':'h5','horizon':5}] if quick else
                    [{'id':'h5','horizon':5},{'id':'h7','horizon':7}],
            'arms':candidates, 'baseline_id':'baseline',
            'development_seeds':[6501,6502] if quick else [6501,6502,6503],
            'holdout_seeds':[6601,6602] if quick else [6601,6602,6603],
            'base_config':{**BASE_HPARAMS,
                           'rollouts':4 if quick else BASE_HPARAMS['rollouts'],
                           'episodes_per_rollout':8 if quick else BASE_HPARAMS['episodes_per_rollout'],
                           'eval_episodes':30 if quick else BASE_HPARAMS['eval_episodes']},
            'primary':'exact_success_within_limit', 'selection':'mean_of_task_means_on_development',
            'tie':'catalog_order_first', 'selected_comparison':'holdout_baseline_vs_selected_only',
            'audit_unit':'per task, independent training seed and arm',
            'fixed_budget':'same_rollouts, episodes, epochs; optimizer steps may differ for full_batch',
            'bootstrap':{'seed':650065,'replicates':500,'confidence':.90},
            'reports_are_descriptive':True}


def validate_protocol(proto: dict) -> None:
    require(type(proto) is dict, '协议必须为dict')
    expected=make_protocol(proto.get('variant')=='quick_smoke')
    require(proto==expected, '协议、种子、消融因素或预算已变化；必须新建实验')
    arms=proto['arms']
    require(len({a['id'] for a in arms})==len(arms), 'arm ID重复')
    base=arms[0]
    require(base['id']==proto['baseline_id'] and base['train_mode']=='accum' and not base['changes'], '缺少基线')
    for arm in arms:
        require(arm['category'] in {'baseline','one_factor','two_factor_interaction','implementation_equivalence'},
                'arm类别非法')
        require(arm['train_mode'] in {'full','mini','accum'},'训练模式非法')
        require(set(arm['changes']) <= set(proto['base_config']) and
                all(proto['base_config'][k] != v for k,v in arm['changes'].items()),
                '消融设置没有真的改变因素')
        changes=len(arm['changes'])+(arm['train_mode']!=base['train_mode'])
        require(changes=={'baseline':0,'one_factor':1,'two_factor_interaction':2,
                          'implementation_equivalence':1}[arm['category']],
                '消融改变的因素数与登记类别不符')
    require(not set(proto['development_seeds']) & set(proto['holdout_seeds']), '开发/测试训练种子泄露')


def config_for(proto: dict, task: dict, arm: dict, seed: int) -> Config:
    validate_protocol(proto)
    require(task in proto['tasks'] and arm in proto['arms'], '任务/arm未登记')
    require(type(seed) is int and seed in proto['development_seeds']+proto['holdout_seeds'], '未登记的训练seed')
    params={**proto['base_config'], **arm['changes'], 'horizon':task['horizon'], 'seed':seed}
    cfg=Config(**params);cfg.validate()
    return cfg


def planned_keys(proto: dict, phase: str, selected: str|None=None) -> set[tuple[str,int,str]]:
    validate_protocol(proto)
    require(phase in {'development','holdout'},'阶段非法')
    if phase=='development':
        arms=[a['id'] for a in proto['arms']]; seeds=proto['development_seeds']
    else:
        require(type(selected) is str and selected in {a['id'] for a in proto['arms'] if a['selectable']},
                '留出阶段只能使用已登记可选arm')
        arms=sorted({proto['baseline_id'],selected});seeds=proto['holdout_seeds']
    return {(task['id'],seed,arm) for task in proto['tasks'] for seed in seeds for arm in arms}

In [15]:
proto=make_protocol()
validate_protocol(proto)
print('任务:',proto['tasks'])
print('开发训练seed:',proto['development_seeds'])
print('留出训练seed:',proto['holdout_seeds'])
for arm in proto['arms']:
    print(arm['id'], arm['category'],'changes=',arm['changes'],
          'mode=',arm['train_mode'],'selection=',arm['selectable'])
print('开发预登记试验数:',len(planned_keys(proto,'development')))
print('若选择一个非基线，留出试验数:',len(planned_keys(proto,'holdout','gae_zero')))

任务: [{'id': 'h5', 'horizon': 5}, {'id': 'h7', 'horizon': 7}]
开发训练seed: [6501, 6502, 6503]
留出训练seed: [6601, 6602, 6603]
baseline baseline changes= {} mode= accum selection= True
gae_zero one_factor changes= {'gae_lambda': 0.0} mode= accum selection= True
value_plain one_factor changes= {'value_method': 'plain'} mode= accum selection= True
no_kl_stop one_factor changes= {'kl_stop_enabled': False} mode= accum selection= True
full_batch one_factor changes= {} mode= full selection= True
micro_unsplit implementation_equivalence changes= {} mode= mini selection= False
gae_value_combo two_factor_interaction changes= {'gae_lambda': 0.0, 'value_method': 'plain'} mode= accum selection= False
开发预登记试验数: 42
若选择一个非基线，留出试验数: 12


## 9．单次试验怎样留下可审计链路

每个 `(phase, task_id, seed, arm_id)` 是一个唯一试验位置。保存：初始化摘要、固定调度摘要、训练轨迹摘要、最终 Actor/Critic 权重、准确主指标与更新预算。

重评分不只是重新读取旧 `report.json`：会从可信本地checkpoint恢复模型，用固定评估seed计算结果，核对权重和trace、optimizer计数。不做精确断点续训；摘要用于检测意外变化，不是签名或恶意文件安全证明。

### 实现单元：逐试验采样、保存、训练轨迹结构审计

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [16]:
def new_run_dir(root: str|Path, label: str) -> Path:
    root=Path(root).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)
    path=root/(label+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    path.mkdir(exist_ok=False)
    return path


def save_json(path: Path, data) -> None:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(data,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf-8')


def save_jsonl(path: Path, rows: list[dict]) -> None:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(''.join(canonical(r)+'\n' for r in rows),encoding='utf-8')


def bounded_json(path: Path, limit: int=8_000_000):
    require(path.is_file() and path.stat().st_size<=limit,'缺失/过大的报告文件')
    def pairs(xs):
        result={}
        for k,v in xs:
            require(k not in result,'重复JSON键')
            result[k]=v
        return result
    def fail(_):raise LessonError('非有限JSON常数')
    try:
        return json.loads(path.read_text(encoding='utf-8'),object_pairs_hook=pairs,parse_constant=fail)
    except (json.JSONDecodeError,UnicodeError,RecursionError) as exc: raise LessonError('报告JSON无效') from exc


def trace_paths(out: Path, phase: str, tid: str, aid: str, seed: int) -> tuple[Path,Path]:
    key=f'{phase}__{tid}__{aid}__seed{seed}'
    return out/'traces'/(key+'.json'),out/'weights'/(key+'.pt')


def run_trial(proto: dict, task: dict, arm: dict, seed: int, phase: str, out: Path) -> dict:
    cfg=config_for(proto,task,arm,seed)
    require((phase=='development' and seed in proto['development_seeds']) or
            (phase=='holdout' and seed in proto['holdout_seeds']), '阶段与训练seed错配')
    torch.set_num_threads(1);seed_all(seed)
    actor=Actor(cfg.actor_hidden).double();critic=Critic(cfg.critic_hidden).double()
    a_init=tensor_hash(actor.state_dict());v_init=tensor_hash(critic.state_dict())
    rng=random.Random(seed+10)
    schedule=[rng.randrange(2**31) for _ in range(cfg.rollouts)]
    trained_actor,trained_critic,trace=train_arm(cfg,arm['train_mode'],schedule,actor,critic)
    evaluated=evaluate_policy(trained_actor,trained_critic,cfg.gamma,cfg.horizon,
                              cfg.eval_episodes,seed=770000+seed*10+cfg.horizon)
    trace_file,weights_file=trace_paths(out,phase,task['id'],arm['id'],seed)
    save_json(trace_file,trace)
    weights_file.parent.mkdir(exist_ok=True,parents=True)
    tensor_states={'actor':{k:v.detach().cpu().clone() for k,v in trained_actor.state_dict().items()},
                   'critic':{k:v.detach().cpu().clone() for k,v in trained_critic.state_dict().items()}}
    checkpoint={'version':VERSION,'proto_sha256':sha256(proto),'phase':phase,
                'task':task,'arm':arm,'seed':seed,'cfg':asdict(cfg),
                'actor_initial_hash':a_init,'critic_initial_hash':v_init,
                'actor':tensor_states['actor'],'critic':tensor_states['critic'],
                'actor_hash':tensor_hash(tensor_states['actor']),
                'critic_hash':tensor_hash(tensor_states['critic'])}
    torch.save(checkpoint,weights_file)
    result={'phase':phase,'status':'completed','task_id':task['id'],'horizon':cfg.horizon,
            'seed':seed,'arm_id':arm['id'],'primary_exact_success':evaluated['exact_success_within_limit'],
            'sampled_successes':evaluated['sampled_successes'],
            'sampled_episodes':evaluated['sampled_episodes'],
            'critic_value_rmse':evaluated['value_rmse'],
            'training_episodes':trace['episodes'],
            'unique_environment_transitions':trace['unique_sampled_steps'],
            'transition_presentations':trace['transition_presentations'],
            'actor_optimizer_steps':trace['actor_optimizer_steps'],
            'critic_optimizer_steps':trace['critic_optimizer_steps'],
            'stopped_rollouts':trace['actor_stopped_rollouts'],
            'schedule_sha256':sha256(schedule),'initial_actor_hash':a_init,
            'initial_critic_hash':v_init,'final_actor_hash':checkpoint['actor_hash'],
            'final_critic_hash':checkpoint['critic_hash'],'trace_sha256':sha256(trace),
            'trace_file':str(trace_file.relative_to(out)),
            'weights_file':str(weights_file.relative_to(out)),
            'eval_seed':770000+seed*10+cfg.horizon}
    return result


def check_trace(trace: dict,cfg: Config) -> dict:
    require(trace['mode'] in {'accum','mini','full'} and trace['rollouts']==cfg.rollouts,
            '训练记录mode/rollouts与预登记不一致')
    require(trace['episodes']==cfg.rollouts*cfg.episodes_per_rollout,
            '训练episode数量不符')
    require(len(trace['history'])==cfg.rollouts,'缺少rollout轨迹')
    total_steps=total_actor=total_critic=total_presentations=0
    for rid,item in enumerate(trace['history']):
        require(item['rollout_id']==rid and item['episodes']==cfg.episodes_per_rollout,
                'rollout编号或episode数量有缺失')
        n=item['steps']
        require(n==sum(item['episode_lengths']) and len(item['episode_lengths'])==cfg.episodes_per_rollout,
                '变长episode不应丢失或重复transition')
        require(all(1<=l<=cfg.horizon for l in item['episode_lengths']),
                '回合长度非法')
        require(item['terminations']+item['truncations']==cfg.episodes_per_rollout,
                '终止与截断必须区分并穷尽')
        require(len(item['epochs'])==cfg.epochs,'epoch数与登记不符')
        for epoch in item['epochs']:
            rows=epoch['batches']
            require(rows and sum(b['size'] for b in rows)==n,
                    '每个epoch必须恰好覆盖全部transition')
            require(all(b['size']>0 and sum(b['micro_sizes'])==b['size'] for b in rows),
                    'micro数据不守恒')
            require(epoch['weighted']['observed_steps']==n and
                    epoch['weighted']['actor_updates']==sum(b['actor_updated'] for b in rows) and
                    epoch['weighted']['critic_updates']==sum(b['critic_updated'] for b in rows),
                    'epoch统计必须从实际minibatch加权汇总')
            total_presentations+=n
        require(item['actor_updates']==sum(e['weighted']['actor_updates'] for e in item['epochs'])
                and item['critic_updates']==sum(e['weighted']['critic_updates'] for e in item['epochs']),
                'Actor/Critic步数与轨迹不符')
        require(item['critic_updates']>=item['actor_updates'],'Critic必须继续训练，Actor停步不允许抹掉')
        total_steps+=n;total_actor+=item['actor_updates'];total_critic+=item['critic_updates']
    require(trace['unique_sampled_steps']==total_steps and trace['transition_presentations']==total_presentations
            and trace['actor_optimizer_steps']==total_actor and trace['critic_optimizer_steps']==total_critic,
            '全局预算与每次rollout轨迹不一致')
    return {'validated_rollouts':len(trace['history']), 'transitions':total_steps,
            'actor_steps':total_actor,'critic_steps':total_critic,
            'transition_presentations':total_presentations,
            'status':'validated_mechanistic_counts_not_statistical_significance'}


def validate_saved_trials(out: Path, proto: dict, records: list[dict]) -> dict:
    validate_protocol(proto)
    for r in records:
        task=next(t for t in proto['tasks'] if t['id']==r['task_id'])
        arm=next(a for a in proto['arms'] if a['id']==r['arm_id'])
        cfg=config_for(proto,task,arm,r['seed'])
        path,weights=trace_paths(out,r['phase'],r['task_id'],r['arm_id'],r['seed'])
        require(path.exists() and weights.exists(), '追溯文件缺失')
        require(path.stat().st_size < 3_000_000 and weights.stat().st_size < 1_000_000,
                '追溯文件超过大小限制')
        trace=bounded_json(path,3_000_000)
        require(sha256(trace)==r['trace_sha256'], '训练轨迹摘要不匹配')
        checked=check_trace(trace,cfg)
        require(checked['transitions']==r['unique_environment_transitions']
                and checked['actor_steps']==r['actor_optimizer_steps']
                and checked['critic_steps']==r['critic_optimizer_steps'], '预算与审计记录不一致')
        blob=torch.load(weights,map_location='cpu',weights_only=True)
        require(blob['version']==VERSION and blob['proto_sha256']==sha256(proto)
                and blob['phase']==r['phase'] and blob['task']==task and blob['arm']==arm
                and blob['cfg']==asdict(cfg) and blob['seed']==r['seed'],
                '模型元数据与协议不匹配')
        require(blob['actor_initial_hash']==r['initial_actor_hash']
                and blob['critic_initial_hash']==r['initial_critic_hash'], '初始化身份不匹配')
        require(tensor_hash(blob['actor'])==blob['actor_hash']==r['final_actor_hash'] and
                tensor_hash(blob['critic'])==blob['critic_hash']==r['final_critic_hash'],
                '权重内容与记录摘要不匹配')
        actor=Actor(cfg.actor_hidden).double();critic=Critic(cfg.critic_hidden).double()
        actor.load_state_dict(blob['actor'],strict=True)
        critic.load_state_dict(blob['critic'],strict=True)
        measured=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,cfg.eval_episodes,r['eval_seed'])
        require(abs(measured['exact_success_within_limit']-r['primary_exact_success'])<1e-12
                and measured['sampled_successes']==r['sampled_successes']
                and abs(measured['value_rmse']-r['critic_value_rmse'])<1e-11,
                '权重重载和锁定评估不一致')
    return {'validated_runs':len(records),'weights_and_traces_checked':True,
            'boundary':'trusted_lesson_generated_files_only_not_training_resume_or_signature'}

## 10．逐因素差值：按同一任务与训练 seed 配对

先在每个 `(任务,seed)` 上计算变化值，之后每个任务对其训练种子等权，再跨任务等权。

\[
\Delta=\frac{1}{|T|}\sum_{t\in T}\frac{1}{|S|}\sum_{s\in S}
(\operatorname{score}_{t,s,\mathrm{arm}}-\operatorname{score}_{t,s,\mathrm{baseline}}).
\]

算法之间分别排序每个得分列表，再当作同一训练seed对照，会丢失配对身份。只报告两个独立均值也会失去协方差信息。

本课对每个任务内部的**训练seed配对差**做 bootstrap，再跨两个教学 horizon 平均。它是小种子描述性区间，不能把“区间刚好排除0”当作普遍显著性保证。[rliable](https://github.com/google-research/rliable)

两个因素同时改变时，还可以记录描述性差中差：

\[
I=(\text{GAE0+plain})-\text{GAE0}-\text{plain}+\text{baseline}.
\]

这不属于逐因素消融，也不等于因果交互效应已经被普遍识别。

### 实现单元：开发集、配对bootstrap、两因素交互与数值精度敏感度

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [17]:
def validate_records(proto: dict, rows: list[dict], phase: str,
                     selected: str|None=None) -> dict[tuple[str,int,str],dict]:
    expected=planned_keys(proto,phase,selected)
    require(type(rows) is list, '记录需要列表')
    result={}
    for row in rows:
        require(type(row) is dict, '试验记录需要dict')
        key=(row.get('task_id'),row.get('seed'),row.get('arm_id'))
        require(all(type(v) in (str,int) for v in key) and key in expected,
                '试验ID/种子/arm或phase不在计划内')
        require(row.get('phase')==phase and row.get('status')=='completed',
                '训练失败必须保留并报告，不可从均值删除')
        require(key not in result, '重复试验/训练种子不允许择优')
        required={'primary_exact_success','critic_value_rmse','training_episodes',
                  'unique_environment_transitions','transition_presentations',
                  'actor_optimizer_steps','critic_optimizer_steps','stopped_rollouts',
                  'sampled_successes','sampled_episodes'}
        require(required<=set(row),'缺失指标/预算/诊断字段')
        for field in ('primary_exact_success','critic_value_rmse'):
            v=row[field]
            require(type(v) in (int,float) and math.isfinite(v) and v>=0,
                    field+'非法数值')
        require(row['primary_exact_success']<=1, '主要评分越界')
        for field in required-{'primary_exact_success','critic_value_rmse'}:
            require(type(row[field]) is int and row[field]>=0, field+'非法整数/计数')
        require(row['sampled_successes']<=row['sampled_episodes'] and
                row['actor_optimizer_steps']<=row['critic_optimizer_steps'] and
                row['transition_presentations']>=row['unique_environment_transitions'],
                '评估或训练预算守恒错误')
        result[key]=row
    require(set(result)==expected,'缺少预登记的训练seed；不允许幸存者偏差')
    return result


def means_by_arm(proto: dict, lookup: dict) -> list[dict]:
    result=[]
    for arm in proto['arms']:
        per_task=[]
        for task in proto['tasks']:
            vals=[lookup[(task['id'],seed,arm['id'])]['primary_exact_success']
                  for seed in proto['development_seeds']]
            per_task.append({'task_id':task['id'],'training_seeds':proto['development_seeds'],
                             'scores':vals,'task_mean':sum(vals)/len(vals)})
        result.append({'arm_id':arm['id'],'category':arm['category'],
                       'selectable':arm['selectable'],
                       'mean_equal_task_weight':sum(t['task_mean'] for t in per_task)/len(per_task),
                       'per_task':per_task})
    return result


def select_development(proto: dict, records: list[dict]) -> dict:
    lookup=validate_records(proto,records,'development')
    summary=means_by_arm(proto,lookup)
    eligible=[r for r in summary if r['selectable']]
    winner=max(eligible,key=lambda r:r['mean_equal_task_weight'])
    return {'selected_arm':winner['arm_id'],'baseline_arm':proto['baseline_id'],
            'development_scores':summary,'criterion':proto['selection'],
            'tie':proto['tie'],'uses_holdout':False,
            'development_records_sha256':sha256(sorted(records,key=lambda r:(r['task_id'],r['seed'],r['arm_id']))),
            'protocol_sha256':sha256(proto)}


def paired_differences(proto: dict, lookup: dict, aid: str) -> list[dict]:
    require(aid in {a['id'] for a in proto['arms']}, '未知消融条件')
    baseline=proto['baseline_id']
    return [{'task_id':task['id'],'seed':seed,
             'baseline':lookup[(task['id'],seed,baseline)]['primary_exact_success'],
             'alternative':lookup[(task['id'],seed,aid)]['primary_exact_success'],
             'difference':lookup[(task['id'],seed,aid)]['primary_exact_success']-
                          lookup[(task['id'],seed,baseline)]['primary_exact_success']}
            for task in proto['tasks'] for seed in proto['development_seeds']]


def numeric(values, name='数值', min_length=1):
    require(type(values) is list and len(values)>=min_length, name+'长度错误')
    require(all(type(x) in (int,float) and math.isfinite(x) for x in values),name+'非有限数值')
    return list(map(float,values))


def quantile(values, q):
    vals=sorted(numeric(values,'percentile'))
    require(type(q) in (float,int) and 0<=q<=1,'percentile范围非法')
    x=q*(len(vals)-1);left=int(math.floor(x));right=int(math.ceil(x))
    return vals[left]*(right-x)+vals[right]*(x-left) if left!=right else vals[left]


def stratified_paired_interval(proto: dict, differences: list[dict],
                                development: bool=True) -> dict:
    """每任务内部成对重采样训练seed，而不是独立重采样两种算法或评估episodes。"""
    tasks=[t['id'] for t in proto['tasks']]
    seeds=proto['development_seeds'] if development else proto['holdout_seeds']
    expected={(task,seed) for task in tasks for seed in seeds}
    require(type(differences) is list, '差值需列表')
    lookup={}
    for row in differences:
        require(type(row) is dict and set(row)=={'task_id','seed','baseline','alternative','difference'},
                '配对结构不符')
        key=(row['task_id'],row['seed'])
        require(key in expected and key not in lookup,'训练种子缺失/重复/越界')
        vals=[row[k] for k in ('baseline','alternative','difference')]
        require(all(type(x) in (int,float) and math.isfinite(x) for x in vals),
                '配对差值非有限')
        require(0<=row['baseline']<=1 and 0<=row['alternative']<=1 and
                abs(row['difference']-(row['alternative']-row['baseline']))<1e-12,
                '配对差值不得从独立排序或报告字段伪造')
        lookup[key]=float(row['difference'])
    require(set(lookup)==expected,'不允许静默删除缺失训练seed')
    point=sum(sum(lookup[(t,s)] for s in seeds)/len(seeds) for t in tasks)/len(tasks)
    count=proto['bootstrap']['replicates'];rng=random.Random(proto['bootstrap']['seed'])
    samples=[]
    for _ in range(count):
        task_means=[]
        for t in tasks:
            task_means.append(sum(lookup[(t,rng.choice(seeds))] for _ in seeds)/len(seeds))
        samples.append(sum(task_means)/len(task_means))
    confidence=proto['bootstrap']['confidence']
    result={'paired_equal_task_mean':point,
            'percentile_interval':[quantile(samples,(1-confidence)/2),quantile(samples,(1+confidence)/2)],
            'confidence_label':confidence,'unit':'training_seed_cluster_within_task',
            'independent_training_seeds_per_task':len(seeds),'tasks':len(tasks),
            'not_significance_claim':True,
            'caveat':'small fixed synthetic tasks and few training seeds; illustrative descriptive interval'}
    return result


def interaction_diagnostic(proto: dict, lookup: dict) -> dict:
    """描述性双因素差中差；因轨迹变化、随机性与少种子，不赋因果普遍性。"""
    required={'baseline','gae_zero','value_plain','gae_value_combo'}
    require(required <= {a['id'] for a in proto['arms']},'缺少交互的四个角')
    rows=[]
    for task in proto['tasks']:
        for seed in proto['development_seeds']:
            t=task['id']
            scores={k:lookup[(t,seed,k)]['primary_exact_success'] for k in required}
            interaction=scores['gae_value_combo']-scores['gae_zero']-scores['value_plain']+scores['baseline']
            rows.append({'task_id':t,'seed':seed,'difference_in_differences':interaction,
                         'scores':scores})
    return {'mean_interaction':sum(r['difference_in_differences'] for r in rows)/len(rows),
            'rows':rows,'interpretation':'descriptive interaction on this task, not automatic causal identification'}


def precision_sensitivity(proto: dict, rows: list[dict]) -> dict:
    """仅展示分数四舍五入对开发选项的敏感度，不改已锁定原始选择。"""
    original=select_development(proto,rows)['selected_arm']
    outcomes={}
    for digits in (3, 4, 6):
        rounded=[{**r,'primary_exact_success':round(r['primary_exact_success'],digits)} for r in rows]
        chosen=select_development(proto,rounded)['selected_arm']
        outcomes[str(digits)]={'selected':chosen,'differs_from_unrounded':chosen!=original}
    near_ceiling=sum(r['primary_exact_success']>.999 for r in rows)
    return {'unrounded_selected':original,'rounded_diagnostics':outcomes,
            'dev_trials_over_0_999':near_ceiling,'dev_trials_total':len(rows),
            'caution':'saturation makes method ranking sensitive to small absolute differences; never retune on holdout'}


def ablation_development(proto: dict, rows: list[dict]) -> dict:
    lookup=validate_records(proto,rows,'development')
    effects=[]
    for arm in proto['arms']:
        if arm['id']==proto['baseline_id']:continue
        paired=paired_differences(proto,lookup,arm['id'])
        effects.append({'arm_id':arm['id'],'label':arm['label'],'category':arm['category'],
                        'changed_factors':{**arm['changes'],
                                           **({'train_mode':arm['train_mode']} if arm['train_mode']!='accum' else {})},
                        'paired':paired,'interval':stratified_paired_interval(proto,paired,True)})
    return {'effects':effects,'interaction':interaction_diagnostic(proto,lookup),
            'precision_sensitivity':precision_sensitivity(proto,rows),
            'cautions':[
                'one factor changes a configured setting, not all downstream state trajectories',
                'full_batch changes optimizer update count as well as batch width; do not isolate batch size alone',
                'micro_unsplit is a numerical equivalence audit, not a general performance gain',
                'interaction is two-factor difference-in-differences, not an additional single-factor arm',
                'dev results select the method and are not heldout evidence']}

## 11．锁定选择再打开留出

本课完整协议使用：2个 horizon × 3个开发训练 seed × 7个条件，共42次开发训练；选定一个可选方案后，只在 2 horizon × 3个**新训练 seed** 上重复基线与所选条件，共12次留出训练。

如果开发选择本身恰好等于基线，留出只需要一个候选的6次运行；差值为零，但不声称证明两种不同算法相等。

主指标使用精确有限时域成功概率，而不是对同一个训练结果采样70次就当作70个独立训练seed。训练seed是重复试验单位。

**常见失败：** 在看过测试后换一个“开发最优”；静默删除一次失败训练；把演示的 `micro_unsplit` 当作独立性能方法；不披露 `full_batch` 更新数改变。下面的报告器会逐项拒绝/标记。

### 实现单元：冻结选择、留出成对对照、预算与重新评分

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [18]:
def compare_holdout(proto: dict, selection: dict, records: list[dict]) -> dict:
    selected=selection['selected_arm']
    lookup=validate_records(proto,records,'holdout',selected)
    base=proto['baseline_id'];paired=[]
    for task in proto['tasks']:
        for seed in proto['holdout_seeds']:
            b=lookup[(task['id'],seed,base)]['primary_exact_success']
            a=lookup[(task['id'],seed,selected)]['primary_exact_success']
            paired.append({'task_id':task['id'],'seed':seed,'baseline':b,
                           'alternative':a,'difference':a-b})
    point=stratified_paired_interval(proto,paired,False)
    return {'selected_arm':selected,'baseline_arm':base,'same_as_baseline':selected==base,
            'paired':paired,'paired_interval':point,
            'baseline_equal_task_mean':sum(r['baseline'] for r in paired)/len(paired),
            'selected_equal_task_mean':sum(r['alternative'] for r in paired)/len(paired),
            'warning':'small fixed synthetic task; CI is descriptive and not a formal significance test'}


def budget_by_phase(rows: list[dict]) -> dict:
    out={}
    for phase in ('development','holdout'):
        subset=[r for r in rows if r['phase']==phase]
        per_arm={}
        for aid in sorted({r['arm_id'] for r in subset}):
            selected=[r for r in subset if r['arm_id']==aid]
            per_arm[aid]={'trained_runs':len(selected),
                'episodes':sum(r['training_episodes'] for r in selected),
                'environment_transitions':sum(r['unique_environment_transitions'] for r in selected),
                'transition_presentations':sum(r['transition_presentations'] for r in selected),
                'actor_optimizer_steps':sum(r['actor_optimizer_steps'] for r in selected),
                'critic_optimizer_steps':sum(r['critic_optimizer_steps'] for r in selected),
                'evaluation_episodes':sum(r['sampled_episodes'] for r in selected),
                'actor_stopped_rollouts':sum(r['stopped_rollouts'] for r in selected)}
        out[phase]=per_arm
    return {'by_phase':out,
            'note':'environment transitions, minibatch presentations, optimizer steps differ; no GPU FLOPs or wall-time equivalence measured'}


def build_report(proto: dict, development_rows: list[dict], selection: dict,
                 holdout_rows: list[dict]) -> dict:
    validate_protocol(proto)
    require(select_development(proto,development_rows)==selection,
            '开发选择冻结后不能看测试重新改选')
    return {'version':VERSION,'variant':proto['variant'],
            'protocol_sha256':sha256(proto),'selection_sha256':sha256(selection),
            'development':ablation_development(proto,development_rows),
            'development_selection':selection,
            'holdout':compare_holdout(proto,selection,holdout_rows),
            'budgets':budget_by_phase(development_rows+holdout_rows),
            'methodology':[
                'holdout training seeds distinct from development seeds',
                'train full dev grid before selecting; freeze selection file before holdout',
                'one-factor changes explicit config only; downstream trajectories diverge',
                'same seed and initial weights within each task, common rollout seed schedule not identical trajectories',
                'primary exact dynamic-programming horizon success probability, not sampled episode rate',
                'no automatic wins, significance or causality extrapolation',
                'interaction & micro-equivalence are diagnostics excluded from selection',
                'no pretrained language model or CUDA benchmark']}


def rescore_run(folder: str|Path, verify_files: bool=True) -> dict:
    folder=Path(folder).expanduser().resolve()
    proto=bounded_json(folder/'protocol.json',100_000);validate_protocol(proto)
    development=bounded_json(folder/'development.json',4_000_000)
    selection=bounded_json(folder/'selection.json',200_000)
    holdout=bounded_json(folder/'holdout.json',4_000_000)
    records=development+holdout
    if verify_files: validate_saved_trials(folder,proto,records)
    report=build_report(proto,development,selection,holdout)
    existing=bounded_json(folder/'report.json',8_000_000)
    require(report==existing,'冻结选择/评分或报告内容不一致')
    return {'status':'validated','trials':len(records),'selection':selection['selected_arm'],
            'report_matches_saved':True,'training_not_repeated':True,
            'source_files_modified':False}


def full_experiment(root='lesson_65_outputs', quick: bool=False) -> Path:
    torch.set_num_threads(1)
    proto=make_protocol(quick);validate_protocol(proto)
    out=new_run_dir(root,proto['variant'])
    manifest={'version':VERSION,'variant':proto['variant'],'status':'running',
              'python':platform.python_version(),'torch':torch.__version__,
              'device':'cpu-double','protocol_sha256':sha256(proto),'sources':SOURCES}
    save_json(out/'manifest.json',manifest);save_json(out/'protocol.json',proto)
    try:
        dev=[]
        for task in proto['tasks']:
            for seed in proto['development_seeds']:
                for arm in proto['arms']:
                    dev.append(run_trial(proto,task,arm,seed,'development',out))
        save_json(out/'development.json',dev)
        # Holdout is not examined until this selection file exists.
        selection=select_development(proto,dev)
        save_json(out/'selection.json',selection)
        holdout=[]
        for task in proto['tasks']:
            for seed in proto['holdout_seeds']:
                for aid in sorted({proto['baseline_id'],selection['selected_arm']}):
                    arm=next(a for a in proto['arms'] if a['id']==aid)
                    holdout.append(run_trial(proto,task,arm,seed,'holdout',out))
        save_json(out/'holdout.json',holdout)
        report=build_report(proto,dev,selection,holdout)
        save_json(out/'report.json',report)
        save_jsonl(out/'all_trials.jsonl',dev+holdout)
        manifest['status']='completed';save_json(out/'manifest.json',manifest)
        print(f"{proto['variant']}: development trials {len(dev)}, heldout trials {len(holdout)}")
        print('Selected by development:',selection['selected_arm'])
        print('Holdout paired difference:',report['holdout']['paired_interval']['paired_equal_task_mean'])
        print('Directory:',out)
        return out
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        save_json(out/'manifest.json',manifest)
        raise

## 12．消融之前先检验故障机制

**数学反例：** 外部截断如果当作真实终止，少了 final observation 的价值；Critic 的三种value loss并不相同。

**选择反例：** 我们另造一套“开发里看起来最好、留出反而差”的合成分数，以检查选择与留出隔离。合成分数不来自模型，不能被报为 PPO 训练结果。

这个程序会同时核对同一微型任务中不同配置的预算，而不是只给最终点估计。

### 实现单元：确定性反例、三个独立练习及验收器

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [19]:
def mechanism_counterexamples() -> dict:
    """确定性数学反例，不捏造多seed训练成绩。"""
    ep=[[Step(0,0,STEP_COST,0,False,True,float(math.log(.5)))]]
    valid=generalized_advantage_estimation(ep,[.2],[.8],.95,.95)
    wrong=generalized_advantage_estimation(ep,[.2],[0.],.95,.95)
    prediction=torch.tensor([3.,1.],dtype=torch.float64,requires_grad=True)
    old=torch.tensor([0.,0.],dtype=torch.float64)
    target=torch.tensor([2.,0.],dtype=torch.float64)
    values={}
    for method in ('plain','clipmax','clipped_pred'):
        l,info=value_loss(prediction,old,target,epsilon=.2,method=method)
        values[method]={'loss':float(l.detach()),'gradient':torch.autograd.grad(l,prediction,retain_graph=True)[0].tolist(),
                        'components':info}
    three_modes={'accum':'17有效transition minibatch，以5步micro累积，不改optimizer步数',
                 'mini':'同一个17-step minibatch直接求导，数值近似同accum',
                 'full':'整个rollout一次优化器更新，optimizer步数一般显著不同'}
    return {'gae_truncation':{'truncated_step':asdict(ep[0][0]),
              'correct_bootstrap_delta':valid['delta'][0],
              'wrong_zero_bootstrap_delta':wrong['delta'][0],
              'correct_lambda_target':valid['lambda_target'][0],
              'wrong_lambda_target':wrong['lambda_target'][0]},
            'value_losses':values,'batch_mode_contract':three_modes,
            'warning':'每个数学例子只是机制诊断，不是某个真实训练arm的成功率'}


def synthetic_selection_counterexample(proto: dict) -> dict:
    """只用合成分数验证选择隔离：开发胜者可以在测试变差。"""
    validate_protocol(proto)
    base=proto['baseline_id']; scores={'baseline':.5,'gae_zero':.65,'value_plain':.51,
      'no_kl_stop':.49,'full_batch':.48,'micro_unsplit':.5,'gae_value_combo':.70}
    def dummy(task,seed,aid,phase,value):
        return {'phase':phase,'status':'completed','task_id':task['id'],
            'horizon':task['horizon'],'seed':seed,'arm_id':aid,
            'primary_exact_success':value,'critic_value_rmse':.1,
            'training_episodes':64,'unique_environment_transitions':200,
            'transition_presentations':400,'actor_optimizer_steps':20,
            'critic_optimizer_steps':20,'stopped_rollouts':0,
            'sampled_successes':10,'sampled_episodes':50}
    dev=[dummy(t,seed,a['id'],'development',scores[a['id']]) for t in proto['tasks']
         for seed in proto['development_seeds'] for a in proto['arms']]
    selection=select_development(proto,dev)
    chosen=selection['selected_arm']
    holdout=[dummy(t,seed,aid,'holdout',.45 if aid==chosen and aid!=base else .5)
             for t in proto['tasks'] for seed in proto['holdout_seeds']
             for aid in sorted({base,chosen})]
    report=build_report(proto,dev,selection,holdout)
    return {'selection':selection['selected_arm'],
            'dev_chosen':next(x['mean_equal_task_weight'] for x in selection['development_scores'] if x['arm_id']==chosen),
            'holdout_diff':report['holdout']['paired_interval']['paired_equal_task_mean'],
            'interval':report['holdout']['paired_interval']['percentile_interval'],
            'notice':'constructed scores; not measured PPO trial outcomes'}


def factors_reference(baseline: dict, alternative: dict) -> dict:
    """只比较预登记字段；返回不同字段，不靠文本标签判断因素。"""
    require(type(baseline) is dict and type(alternative) is dict and
            set(baseline)==set(alternative), '两个配置应具备相同字段')
    return {k:{'before':baseline[k],'after':alternative[k]}
            for k in sorted(baseline) if baseline[k]!=alternative[k]}


def paired_delta_reference(rows: list[dict], baseline_id: str, alternative_id: str) -> float:
    require(type(rows) is list and len(rows)>=4 and baseline_id!=alternative_id, '至少两组完整配对')
    lookup={}
    for r in rows:
        require(type(r) is dict and set(r)=={'task_id','seed','arm_id','score'}, '配对结构非法')
        require(type(r['task_id']) is str and type(r['seed']) is int and
                r['arm_id'] in {baseline_id,alternative_id} and
                type(r['score']) in (int,float) and math.isfinite(r['score']) and 0<=r['score']<=1,
                '配对字段非法')
        key=(r['task_id'],r['seed'],r['arm_id'])
        require(key not in lookup, '重复配对记录')
        lookup[key]=r['score']
    groups={(t,s) for t,s,_ in lookup}
    require(len(groups)>=2 and set(lookup)=={(t,s,a) for t,s in groups for a in (baseline_id,alternative_id)},
            '不能丢弃缺失的baseline/alternative')
    tasks=sorted({t for t,s in groups})
    require(len({s for t,s in groups if t==tasks[0]})>=2,'至少两个独立seed')
    require(all({s for t,s in groups if t==t0}=={s for t,s in groups if t==tasks[0]} for t0 in tasks),
            '不同任务必须有相同训练seed清单，不能静默删掉')
    differences=[]
    for t in tasks:
        seeds=sorted({s for task,s in groups if task==t})
        differences.append(sum(lookup[(t,s,alternative_id)]-lookup[(t,s,baseline_id)] for s in seeds)/len(seeds))
    return sum(differences)/len(differences)


def audit_budget_reference(trace: dict) -> dict:
    require(type(trace) is dict and type(trace.get('history')) is list and trace['history'],
            '轨迹缺失')
    transitions=presentations=actor_updates=critic_updates=0
    for item in trace['history']:
        require(type(item) is dict and type(item.get('steps')) is int and item['steps']>0,
                '无效rollout')
        require(type(item.get('episode_lengths')) is list and
                sum(item['episode_lengths'])==item['steps'], 'episode长度之和与step不符')
        transitions+=item['steps']
        require(type(item.get('epochs')) is list and item['epochs'], '缺少epoch')
        for epoch in item['epochs']:
            batches=epoch['batches']
            require(batches and sum(b['size'] for b in batches)==item['steps'],
                    '一个epoch少算了transition')
            presentations+=sum(b['size'] for b in batches)
            actor_updates+=sum(bool(b['actor_updated']) for b in batches)
            critic_updates+=sum(bool(b['critic_updated']) for b in batches)
    return {'unique_transitions':transitions,'transition_presentations':presentations,
            'actor_updates':actor_updates,'critic_updates':critic_updates}


def check_exercise_a(fn) -> dict:
    base={'gae_lambda':.95,'value_method':'clipmax','train_mode':'accum'}
    changed={**base,'gae_lambda':0.0}
    require(fn(base,changed)=={'gae_lambda':{'before':.95,'after':0.0}}, '必须只报告真正变化的一项')
    combo={**base,'gae_lambda':0.0,'value_method':'plain'}
    require(len(fn(base,combo))==2, '两因素交互不能叫单因素')
    require(fn(base,base)=={}, '完全相同不应算因素')
    try:fn(base,{'gae_lambda':0.0})
    except (ValueError,TypeError):pass
    else:raise LessonError('不能缺少基线字段还继续比较')
    return {'exercise':'A','passed':True}


def check_exercise_b(fn) -> dict:
    rows=[{'task_id':t,'seed':seed,'arm_id':aid,'score':score}
          for t in ('h5','h7') for seed in (1,2,3)
          for aid,score in [('baseline',.4),('new',.5 if t=='h5' else .6)]]
    expected=.15
    require(abs(fn(rows,'baseline','new')-expected)<1e-12, '应先逐seed/任务配对求差，再按任务等权')
    require(abs(fn(list(reversed(rows)),'baseline','new')-expected)<1e-12,'不能依赖行排序')
    for bad in (rows[:-1],rows+[rows[0]],
                [dict(r,score=float('nan')) if i==0 else r for i,r in enumerate(rows)]):
        try:fn(bad,'baseline','new')
        except (ValueError,TypeError):pass
        else:raise LessonError('不能静默过滤缺失、重复或非有限训练seed')
    return {'exercise':'B','passed':True}


def check_exercise_c(fn) -> dict:
    trace={'history':[{'steps':3,'episode_lengths':[2,1],
                       'epochs':[{'batches':[{'size':2,'actor_updated':True,'critic_updated':True},
                                             {'size':1,'actor_updated':False,'critic_updated':True}]},
                                 {'batches':[{'size':3,'actor_updated':True,'critic_updated':True}]}]},
                      {'steps':2,'episode_lengths':[2],
                       'epochs':[{'batches':[{'size':2,'actor_updated':True,'critic_updated':True}]}]}]}
    result={'unique_transitions':5,'transition_presentations':8,
            'actor_updates':3,'critic_updates':4}
    require(fn(trace)==result,'不能把呈现次数当成新环境步数，Actor/Critic步数要分开')
    bad=copy.deepcopy(trace);bad['history'][0]['epochs'][0]['batches'].pop()
    try:fn(bad)
    except (ValueError,TypeError):pass
    else:raise LessonError('minibatch丢transition必须拒绝')
    return {'exercise':'C','passed':True}

In [20]:
mechanisms=mechanism_counterexamples()
print('截断保留/丢失V(final):',mechanisms['gae_truncation']['correct_bootstrap_delta'],
      mechanisms['gae_truncation']['wrong_zero_bootstrap_delta'])
for name,detail in mechanisms['value_losses'].items():
    print('value',name,detail['loss'],detail['gradient'])
print('合成反例(不是PPO实测):',synthetic_selection_counterexample(make_protocol(True)))

截断保留/丢失V(final): 0.55 -0.21000000000000002
value plain 0.5 [0.5, 0.5]
value clipmax 1.06 [0.0, 0.5]
value clipped_pred 0.8200000000000001 [0.0, 0.0]
合成反例(不是PPO实测): {'selection': 'gae_zero', 'dev_chosen': 0.65, 'holdout_diff': -0.04999999999999999, 'interval': [-0.04999999999999999, -0.04999999999999999], 'notice': 'constructed scores; not measured PPO trial outcomes'}


## 13．独立练习：先填空再开检查

练习不执行训练模型，也不修改前面的辅助实现。`RUN_EXERCISE_*` 默认关闭，单元顺序成功不代表独立练习已完成。

**练习 A：精确消融字段 diff。** 对相同字段集合的基线与新配置返回变更字典；同一配置返回空，两个变化不能偷偷只返回一个；缺字段要拒绝。

In [21]:
RUN_EXERCISE_A=False
RUN_EXERCISE_B=False
RUN_EXERCISE_C=False

def changed_fields_ex(baseline,alternative):
    raise NotImplementedError('请逐字段比较配置，并检查字段集合一致')

if RUN_EXERCISE_A:
    print(check_exercise_a(changed_fields_ex))

**练习 B：配对效果。** 对每个任务/训练seed同时读取基线和候选，计算差值。要求同一训练seed清单完整、拒绝NaN/重复行；每任务先取seed平均，再按任务平均，不随行顺序变化。

In [22]:
def paired_effect_ex(rows,baseline_id,alternative_id):
    raise NotImplementedError('请实现任务内训练seed成对均值')

if RUN_EXERCISE_B:
    print(check_exercise_b(paired_effect_ex))

**练习 C：预算审计。** 一个训练rollout的 `steps` 是新环境步数；每epoch重复呈现这些step，minibatch各一次。真实 Actor/Critic 更新数要按各自是否更新计数。出现缺失 minibatch 要拒绝。

In [23]:
def audit_budget_ex(trace):
    raise NotImplementedError('请分开计数environment steps, presentations和optimizer updates')

if RUN_EXERCISE_C:
    print(check_exercise_c(audit_budget_ex))

## 14．参考答案

请先完成自己的练习，再对照下面已有的参考函数。它们不覆盖你的函数。

In [24]:
print(check_exercise_a(factors_reference))
print(check_exercise_b(paired_delta_reference))
print(check_exercise_c(audit_budget_reference))

{'exercise': 'A', 'passed': True}
{'exercise': 'B', 'passed': True}
{'exercise': 'C', 'passed': True}


## 15．固定检查与故障注入

固定检查不仅覆盖正常收敛，还核对合同不被改动、同来源seed初始化一致、不同seed确实不同、GAE截断、value loss梯度、微批累积、KL停Actor而Critic继续，以及逐试验重载和训练轨迹篡改拒绝。

下面还会运行多个**实际会让对应检查失败**的目标故障版本：静默删除seed、错把全部环境步重复呈现算成新数据、把截断当真实终止、偷换critic损失、关闭KL门控后仍声称启用等。

“某个故障被检查抓住”不等于已经穷尽代码漏洞，更不等于对真实深度RL系统完成安全审计。

### 实现单元：固定检查、数值等价性与故障注入

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [25]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,actual,expected,tol=1e-10):
        self.ok(name,abs(float(actual)-float(expected))<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError,KeyError):self.names.append(name);return
        raise AssertionError(name+': 本应拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    c=Checks();full=make_protocol();quick=make_protocol(True)
    validate_protocol(full);validate_protocol(quick)
    c.ok('版本不同',full['variant']!=quick['variant'])
    c.ok('默认二任务',len(full['tasks'])==2)
    c.ok('开发三独立训练seed',len(full['development_seeds'])==3)
    c.ok('留出三全新训练seed',len(full['holdout_seeds'])==3)
    c.ok('开发留出seed完全不重叠',not set(full['development_seeds'])&set(full['holdout_seeds']))
    c.ok('目录七组条件',len(full['arms'])==7)
    c.ok('逐因素四组单更改',sum(a['category']=='one_factor' for a in full['arms'])==4)
    c.ok('微批等价诊断不参与选择',not next(a for a in full['arms'] if a['id']=='micro_unsplit')['selectable'])
    c.ok('双因素交互不参与选择',not next(a for a in full['arms'] if a['id']=='gae_value_combo')['selectable'])
    c.ok('开发网格规模42',len(planned_keys(full,'development'))==42)
    c.ok('baseline与另一个arm留出规模12',len(planned_keys(full,'holdout','gae_zero'))==12)
    c.ok('选中baseline留出规模6',len(planned_keys(full,'holdout','baseline'))==6)
    for arm in full['arms']:
        cfg=config_for(full,full['tasks'][0],arm,full['development_seeds'][0]);cfg.validate()
        c.ok('条件合法_'+arm['id'],True)
    for field,value in [('tasks',[{'id':'h5','horizon':3}]),
                        ('base_config',{'rollouts':1}),
                        ('development_seeds',[6601,6502,6503]),
                        ('arms',ablation_catalog()[:-1])]:
        bad=copy.deepcopy(full);bad[field]=value
        c.rejects('预登记不允许改_'+field,lambda bad=bad:validate_protocol(bad))
    invalid_cfg=Config(value_method='unknown')
    c.rejects('value方法未知拒绝',invalid_cfg.validate)
    invalid_cfg=Config(kl_stop_enabled=1)
    c.rejects('KL布尔字段类型严格',invalid_cfg.validate)
    invalid_cfg=Config(bootstrap_truncation=1)
    c.rejects('截断开关不能用整数代替',invalid_cfg.validate)
    math_report=mechanism_counterexamples()
    trunc=math_report['gae_truncation']
    c.close('截断应保留final_obs value',trunc['correct_bootstrap_delta'],.55)
    c.close('错误将截断当terminal',trunc['wrong_zero_bootstrap_delta'],-.21)
    c.close('lambda target保留bootstrap',trunc['correct_lambda_target'],.75)
    v=math_report['value_losses']
    c.ok('value plain与clipmax不等价',v['plain']['loss']!=v['clipmax']['loss'])
    c.ok('clipped_pred与clipmax不等价',v['clipped_pred']['loss']!=v['clipmax']['loss'])
    c.ok('clipmax与plain梯度不同',v['plain']['gradient']!=v['clipmax']['gradient'])
    c.ok('三种value目标都明确',set(v)=={'plain','clipmax','clipped_pred'})
    c.close('IQM测试值只用来解释不是训练结果',sum([0,0,0,1,100])/5,20.2)
    c.close('按实际transition数加权',(.4*2+.8*3)/5,.64)
    seed_all(6501)
    actor=Actor(24).double();critic=Critic(24).double()
    cfg=Config(seed=6501,horizon=5,rollouts=2,episodes_per_rollout=9,epochs=2,
               minibatch_size=17,microbatch_size=5)
    eps=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(6503))
    buf=make_buffer(actor,critic,eps,cfg,0)
    c.ok('buffer保留原始优势和冻结归一化优势',buf['advantages'].shape==buf['raw_advantages'].shape)
    c.ok('rollout初始旧策略KL为零',rollout_diagnostics(actor,critic,buf,cfg)['old_state_kl_exact_visited']<1e-10)
    order=shuffled_minibatches(buf['sampled_steps'],17,stable_epoch_seed(6501,0,0))
    ids=[i for part in order for i in part]
    c.ok('每个step恰好覆盖一次',sorted(ids)==list(range(buf['sampled_steps'])))
    mini=order[0]
    sig=frozen_signature(buf)
    full_grad=full_minibatch_gradients(actor,critic,buf,mini,cfg,17)
    actor_grads=[p.grad.detach().clone() for p in actor.parameters()]
    critic_grads=[p.grad.detach().clone() for p in critic.parameters()]
    micro_grad=full_minibatch_gradients(actor,critic,buf,mini,cfg,5)
    c.ok('same有效minibatch累计等价Actor',all(torch.allclose(a,p.grad,atol=1e-10,rtol=1e-10)
             for a,p in zip(actor_grads,actor.parameters())))
    c.ok('same有效minibatch累计等价Critic',all(torch.allclose(a,p.grad,atol=1e-10,rtol=1e-10)
             for a,p in zip(critic_grads,critic.parameters())))
    c.ok('micro累计不修改冻结rollout',frozen_signature(buf)==sig)
    c.ok('micro实际大小加起来是mini',sum(micro_grad['micro_sizes'])==len(mini))
    c.ok('full与micro损失统计近似一致',abs(full_grad['actor_loss_before']-micro_grad['actor_loss_before'])<1e-10)
    # Critic可继续更新，即便Actor因 KL gate 停止。
    tiny=Config(seed=6501,horizon=5,rollouts=2,episodes_per_rollout=9,epochs=2,
                target_kl=1e-12,minibatch_size=17,microbatch_size=5)
    mutated_actor=copy.deepcopy(actor);mutated_critic=copy.deepcopy(critic)
    with torch.no_grad():
        p=next(mutated_actor.parameters());p[0,0]+=0.15
    buf['expected_actor_hash']=tensor_hash(mutated_actor.state_dict())
    old_actor=tensor_hash(mutated_actor.state_dict());old_v=tensor_hash(mutated_critic.state_dict())
    aopt=torch.optim.Adam(mutated_actor.parameters(),lr=.01)
    vopt=torch.optim.Adam(mutated_critic.parameters(),lr=.01)
    step=minibatch_update(mutated_actor,mutated_critic,buf,mini,tiny,aopt,vopt,5)
    c.ok('人为越过阈值真的停止Actor',not step['actor_updated'] and step['actor_stopped'])
    c.ok('Actor停止后Critic仍更新',step['critic_updated'])
    c.ok('Actor停止后权重没有改变',tensor_hash(mutated_actor.state_dict())==old_actor)
    c.ok('Critic停止期间仍有进展',tensor_hash(mutated_critic.state_dict())!=old_v)
    c.ok('冻结rollout seal仍有效',buf['seal']==frozen_signature(buf))
    cfg_off=Config(seed=6501,horizon=5,rollouts=2,episodes_per_rollout=9,epochs=2,
                   target_kl=1e-12,kl_stop_enabled=False)
    c.ok('关掉KL停止属于单因素',factors_reference(asdict(tiny),asdict(cfg_off))==
         {'kl_stop_enabled':{'before':True,'after':False}})
    fake=synthetic_selection_counterexample(quick)
    c.ok('合成选择严格只看开发',fake['selection']=='gae_zero')
    c.close('构造测试表现反转但不改选择',fake['holdout_diff'],-.05)
    c.close('常数配对差的重采样端点',fake['interval'][0],-.05)
    c.close('常数配对差的重采样另一端',fake['interval'][1],-.05)
    c.ok('两任务不同seed bootstrap标签说明独立单位',
         stratified_paired_interval(full,[{'task_id':t['id'],'seed':s,'baseline':.3,
           'alternative':.4,'difference':.1} for t in full['tasks'] for s in full['development_seeds']])['unit']==
         'training_seed_cluster_within_task')
    c.rejects('配对CI不能缺训练seed',lambda:stratified_paired_interval(full,[
        {'task_id':t['id'],'seed':s,'baseline':.3,'alternative':.4,'difference':.1}
        for t in full['tasks'] for s in full['development_seeds']][:-1]))
    c.rejects('配对差值不能伪造',lambda:stratified_paired_interval(full,[
        {'task_id':t['id'],'seed':s,'baseline':.3,'alternative':.4,'difference':.2}
        for t in full['tasks'] for s in full['development_seeds']]))
    c.ok('练习A参考',check_exercise_a(factors_reference)['passed'])
    c.ok('练习B参考',check_exercise_b(paired_delta_reference)['passed'])
    c.ok('练习C参考',check_exercise_c(audit_budget_reference)['passed'])
    # 真实独立训练种子：文件追溯、可复现与事件归因。
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp);test_proto=make_protocol(True)
        test_task=test_proto['tasks'][0];baseline=test_proto['arms'][0]
        row=run_trial(test_proto,test_task,baseline,6501,'development',path)
        row_repeat=run_trial(test_proto,test_task,baseline,6501,'development',path/'repeat')
        for field in ('primary_exact_success','sampled_successes','critic_value_rmse',
                      'trace_sha256','initial_actor_hash','final_actor_hash','final_critic_hash',
                      'actor_optimizer_steps','critic_optimizer_steps'):
            c.ok('独立重复同seed_'+field,row[field]==row_repeat[field])
        other=run_trial(test_proto,test_task,baseline,6502,'development',path)
        c.ok('不同训练seed初始化不同',row['initial_actor_hash']!=other['initial_actor_hash'])
        alt=next(a for a in test_proto['arms'] if a['id']=='micro_unsplit')
        equivalence=run_trial(test_proto,test_task,alt,6501,'development',path)
        c.ok('同seed条件初始权重一致',row['initial_actor_hash']==equivalence['initial_actor_hash'])
        c.ok('不改变有效minibatch则optimizer步数一致',
             row['actor_optimizer_steps']==equivalence['actor_optimizer_steps']
             and row['critic_optimizer_steps']==equivalence['critic_optimizer_steps'])
        c.ok('数值等价条件最终任务评分近似一致',
             abs(row['primary_exact_success']-equivalence['primary_exact_success'])<1e-10)
        report=validate_saved_trials(path,test_proto,[row,other,equivalence])
        c.ok('真实试验权重与轨迹通过审计',report['validated_runs']==3)
        trace_path=path/row['trace_file'];original=bounded_json(trace_path)
        trace=copy.deepcopy(original);trace['history'][0]['steps']+=1
        save_json(trace_path,trace)
        c.rejects('训练报告篡改steps被文件审计拒绝',lambda:validate_saved_trials(path,test_proto,[row]))
        save_json(trace_path,original)
        trace=copy.deepcopy(original);trace['history'][0]['epochs'][0]['batches'][0]['size']+=1
        c.rejects('轨迹短尾计数错误被审计拒绝',lambda:check_trace(trace,config_for(test_proto,test_task,baseline,6501)))
    return {'count':len(c.names),'names':c.names,
            'scope':'CPU三状态独立种子+机制审计；不对小样本声明普适显著性'}



def mutation_checks() -> dict:
    cases=[]
    def detected(name,call):
        try:call()
        except (AssertionError,LessonError,ValueError,TypeError):cases.append({'fault':name,'detected':True})
        else:raise AssertionError('此错误未被原验收捕获：'+name)
    def mistaken_fields(base,alternative):
        return {k:{'before':base[k],'after':alternative[k]} for k in base if base[k]!=alternative[k]}
    # 顺序本身不被检查时也需要具体构造“错误”而非把相同功能算作故障。
    def only_one_field(base,alternative):
        return dict(list(mistaken_fields(base,alternative).items())[:1])
    detected('两因素交互误写成单因素',lambda:check_exercise_a(only_one_field))
    def wrong_pairing(rows,b,a):
        # 删除末尾的seed，再报告好看的均值。
        return paired_delta_reference(rows[:-2],b,a)
    detected('静默删除训练seed',lambda:check_exercise_b(wrong_pairing))
    def wrong_budget(trace):
        r=audit_budget_reference(trace)
        r['unique_transitions']=r['transition_presentations']
        return r
    detected('把重复呈现次数当新环境步',lambda:check_exercise_c(wrong_budget))
    def wrong_bootstrap():
        ep=[[Step(0,0,STEP_COST,0,False,True,float(math.log(.5)))]]
        true=generalized_advantage_estimation(ep,[.2],[.8],.95,.95)['delta'][0]
        wrong=generalized_advantage_estimation(ep,[.2],[0.],.95,.95)['delta'][0]
        require(abs(true-wrong)<1e-12,'截断当terminal导致delta错误')
    detected('截断时丢失final_obs bootstrap',wrong_bootstrap)
    def wrong_value_loss():
        p=torch.tensor([3.,1.],dtype=torch.float64)
        old=torch.zeros(2,dtype=torch.float64);target=torch.tensor([2.,0.],dtype=torch.float64)
        correct=value_loss(p,old,target,method='clipmax')[0]
        wrong=value_loss(p,old,target,method='plain')[0]
        require(abs(float(correct-wrong))<1e-12,'实际启用了错误的critic loss')
    detected('把value clipmax误当plain',wrong_value_loss)
    def wrong_kl_gate():
        cfg=Config(target_kl=1e-12,kl_stop_enabled=False)
        require(cfg.kl_stop_enabled,'KL诊断仍按正常门控停止Actor')
    detected('关闭KL停止却报告已启用',wrong_kl_gate)
    def wrong_selection():
        proto=make_protocol(True)
        synth=synthetic_selection_counterexample(proto)
        require(synth['holdout_diff']>=0,'开发胜者被宣称在测试一定胜出')
    detected('以开发集表现替代留出对照',wrong_selection)
    def wrong_micro_weight():
        sizes=[5,5,2];means=[.1,.1,.9]
        correct=sum(a*b for a,b in zip(sizes,means))/sum(sizes)
        wrong=sum(means)/len(means)
        require(abs(correct-wrong)<1e-12,'把短尾microbatch等权')
    detected('短尾microbatch等权',wrong_micro_weight)
    return {'count':len(cases),'cases':cases,
            'boundary':'targeted faults really rejected; not exhaustive correctness proof'}

In [26]:
CHECKS=run_checks()
MUTATIONS=mutation_checks()
print('固定检查:',CHECKS['count'])
print('故障检测:',MUTATIONS['count'])
for item in MUTATIONS['cases']:
    print(' -',item['fault'],item['detected'])

固定检查: 75
故障检测: 8
 - 两因素交互误写成单因素 True
 - 静默删除训练seed True
 - 把重复呈现次数当新环境步 True
 - 截断时丢失final_obs bootstrap True
 - 把value clipmax误当plain True
 - 关闭KL停止却报告已启用 True
 - 以开发集表现替代留出对照 True
 - 短尾microbatch等权 True


## 16．完整多种子消融：不改方案直接运行

下面默认执行全部注册试验。每个 `(task, seed, arm)` 训练新的 Actor/Critic，不从其他分支续训；相同seed初始参数与随机调度相同，但之后的状态轨迹允许自然分叉。

开发成绩先独立保存为 `development.json`，写出 `selection.json` 后才运行新的留出训练seed。**不会根据留出成绩回头挑其他消融条件。**

本课默认只有两个相关 horizon、六个训练seed/任务（3开发＋3测试）。置信区间仅作描述；近乎满分时各候选极小差异也可能决定开发选择，因此会计算“将分数四舍五入后，选择是否变化”的敏感度报告。

### 实现单元：独立CLI、实验执行及文件校验

完整代码内嵌于下面的可折叠代码单元，**不读取同名脚本或前几课文件**。按顺序执行后，再观察随后的短实验。

In [27]:
def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    modes=parser.add_mutually_exclusive_group()
    modes.add_argument('--checks-only',action='store_true')
    modes.add_argument('--math-only',action='store_true')
    modes.add_argument('--quick',action='store_true',help='缩小任务/seed预算，报告单独标明')
    modes.add_argument('--rescore-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_65_outputs'))
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.rescore_run:
        print(json.dumps(rescore_run(args.rescore_run),ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['count']}项；PPO三状态/两动作消融与审计，不是LLM成绩。")
    if args.checks_only:return 0
    if args.math_only:
        out=new_run_dir(args.output_dir,'math')
        save_json(out/'mechanisms.json',mechanism_counterexamples())
        save_json(out/'checks.json',checks)
        save_json(out/'synthetic_selection.json',synthetic_selection_counterexample(make_protocol(True)))
        save_json(out/'mutations.json',mutation_checks())
        print('目录:',out)
        return 0
    out=full_experiment(args.output_dir,quick=args.quick)
    save_json(out/'checks.json',checks)
    save_json(out/'mutations.json',mutation_checks())
    return 0

In [28]:
RUN_DIRECTORY=None
if RUN_FULL_EXPERIMENT:
    RUN_DIRECTORY=full_experiment(OUTPUT_ROOT, quick=False)
    save_json(RUN_DIRECTORY/'checks.json',CHECKS)
    save_json(RUN_DIRECTORY/'mutations.json',MUTATIONS)
else:
    print('已跳过训练，不编造实测结果。')

registered_ablation: development trials 42, heldout trials 12
Selected by development: no_kl_stop
Holdout paired difference: -4.7871175681779565e-05
Directory: /mnt/data/_lesson65_work/notebook_run/lesson_65_outputs/registered_ablation_20261103T093327_676428Z


In [29]:
if RUN_DIRECTORY is not None:
    REPORT=bounded_json(RUN_DIRECTORY/'report.json')
    print('预登记开发选择:',REPORT['development_selection']['selected_arm'])
    print('各候选开发任务等权成功概率：')
    for entry in REPORT['development_selection']['development_scores']:
        print(entry['arm_id'],round(entry['mean_equal_task_weight'],8),
              '可选择=',entry['selectable'])
    print('留出配对差：',REPORT['holdout']['paired_interval']['paired_equal_task_mean'])
    print('描述区间：',REPORT['holdout']['paired_interval']['percentile_interval'])
    print('开发分数精度敏感度：',REPORT['development']['precision_sensitivity'])

预登记开发选择: no_kl_stop
各候选开发任务等权成功概率：
baseline 0.99999949 可选择= True
gae_zero 0.99999318 可选择= True
value_plain 0.99999933 可选择= True
no_kl_stop 0.99999954 可选择= True
full_batch 0.9999565 可选择= True
micro_unsplit 0.99999949 可选择= False
gae_value_combo 0.99999223 可选择= False
留出配对差： -4.7871175681779565e-05
描述区间： [-9.482717367663059e-05, -9.151437192119378e-07]
开发分数精度敏感度： {'caution': 'saturation makes method ranking sensitive to small absolute differences; never retune on holdout', 'dev_trials_over_0_999': 42, 'dev_trials_total': 42, 'rounded_diagnostics': {'3': {'differs_from_unrounded': True, 'selected': 'baseline'}, '4': {'differs_from_unrounded': True, 'selected': 'baseline'}, '6': {'differs_from_unrounded': True, 'selected': 'baseline'}}, 'unrounded_selected': 'no_kl_stop'}


### 预算与实际更新次数必须一起看

本课固定 rollouts/episodes/epochs，但不保证不同条件的 optimizer step 数相同：

- 关闭KL门控可能让 Actor 更新更多次。
- `full_batch` 每个epoch只做一次全局更新，通常大幅减少优化器step数。
- `micro_unsplit` 保持有效minibatch不变，验证的是同样梯度的不同拆分执行。

因此，比较 `full_batch` 与基线是“整套batching方式”的对照，**不能把差异归因于batch宽度一个变量，同时声称optimizer预算已完全匹配**。

环境transition、训练步骤重复呈现、actor/critic updates与评估episodes应分别报告；这里没有计算GPU FLOPs、显存峰值或机器时延。

In [30]:
if RUN_DIRECTORY is not None:
    budgets=REPORT['budgets']['by_phase']['development']
    print('条件           环境新步   重复呈现   Actor步  Critic步   KL停rollout')
    for aid,r in budgets.items():
        print(f"{aid:16}{r['environment_transitions']:8d}"
              f"{r['transition_presentations']:11d} {r['actor_optimizer_steps']:9d}"
              f"{r['critic_optimizer_steps']:10d} {r['actor_stopped_rollouts']:8d}")
    print('交互差中差：',REPORT['development']['interaction']['mean_interaction'])

条件           环境新步   重复呈现   Actor步  Critic步   KL停rollout
baseline            2134       4268       173       316       31
full_batch          2147       4294        84        96       12
gae_value_combo     2264       4528       163       324       33
gae_zero            2282       4564       153       326       36
micro_unsplit       2134       4268       173       316       31
no_kl_stop          2083       4166       312       312        0
value_plain         2132       4264       169       318       32
交互差中差： -7.925463829846121e-07


## 17．重新读取文件并验证报告，不覆盖旧结果

每个试验有模型 checkpoint 和训练轨迹。`--rescore-run` 使用可信本地文件，验证协议、文件大小、记录数、轨迹一致性、权重摘要、固定评估结果、开发选择和留出报告。它**不会重新训练**，也不相信原报告中没有检验的 `passed` 字段。

下面的调用也应不修改任何输入文件。摘要是误改检测手段，不是数字签名；`torch.load(weights_only=True)` 也不是任意外部文件的全面安全防护。

In [31]:
if RUN_DIRECTORY is not None:
    print(rescore_run(RUN_DIRECTORY))
    print('报告目录：',RUN_DIRECTORY.resolve())

{'status': 'validated', 'trials': 54, 'selection': 'no_kl_stop', 'report_matches_saved': True, 'training_not_repeated': True, 'source_files_modified': False}
报告目录： /mnt/data/_lesson65_work/notebook_run/lesson_65_outputs/registered_ablation_20261103T093327_676428Z


## 18．独立脚本与本课验收

```bash
python lesson_65_ppo_ablations_and_failure_attribution.py --checks-only
python lesson_65_ppo_ablations_and_failure_attribution.py --math-only
python lesson_65_ppo_ablations_and_failure_attribution.py
python lesson_65_ppo_ablations_and_failure_attribution.py --quick
python lesson_65_ppo_ablations_and_failure_attribution.py --rescore-run lesson_65_outputs/实际registered_ablation目录
```

其中 `--quick` 明确标记为小预算烟雾测试，不能混入完整消融的统计结果。

你应该能回答：

1. `GAE λ=0` 是否意味着没有优势估计？
2. `plain`、`clipmax`、`clipped_pred` 为什么不同？
3. 关闭KL停止后，Actor更新数变多；如何控制其他预算？
4. 只改microbatch拆分为何应得到相近梯度，但不代表显存/吞吐已被测试？
5. 为什么“开发胜者”在新训练seed上可能退步？
6. 交互双因素结果为何不能当成第5个单因素？
7. 近乎100%成功时，四舍五入后选项变了说明什么？
8. 怎样从已保存的训练轨迹核对环境步数、实际Actor/Critic步数与指标？

**下一课第66课：从PPO机制到LLM后训练的工程边界。** 将状态/动作、回报、在线采样、价值模型和KL迁移到token级语言模型，核对EOS、长序列、组奖励和GPU预算。

## 原始参考与适用边界

- [PPO 原论文](https://arxiv.org/abs/1707.06347) 与 [Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)
- [GAE](https://arxiv.org/abs/1506.02438) 与 [Gymnasium Time Limit](https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/)
- [Implementation Matters in Deep Policy Gradients](https://arxiv.org/abs/2005.12729)：工程细节可能改变算法行为，不可忽略。
- [Stable-Baselines3 PPO value clipping 源码](https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html)：本课明确对照不同值损失。
- [Deep Reinforcement Learning That Matters](https://ojs.aaai.org/index.php/AAAI/article/view/11694) 与 [rliable](https://github.com/google-research/rliable)：多种子评测、统计不确定性与报告完整性。
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)：固定seed不保证跨后端逐位相同。

本课只针对公开小型CPU环境。两个horizon属于相关任务，少数训练seed与百分位bootstrap不能支持普遍算法显著性；没有自然语言模型、CUDA、3060显存或RL泛化 benchmark 数据。